## Funding analysis (2020-2025)

Computes the standard breakdowns needed for reporting, exported as CSVs sized to paste directly
into the existing Google Sheets template (`raw_data/Funding analysis 2026.xlsx`, 'Europe wide'
tab) - keeps chart formatting consistent with prior reports while the underlying numbers are
computed in code.

Methodology confirmed against that template rather than assumed:
- Funding figure throughout = `'Gov & NP contribution (EUR)'`.
- Excludes any grant whose `'Award purpose'` is *exactly* `'Equipment and infrastructure'` (not
  combined with anything else).
- "Country" = `'Funder Country'` (whose money it is), not `'PI organisation country'`.
- Multi-value cells (countries, pillars, end products) split the grant's funding evenly across
  the **distinct** values listed.
- Growth rate = simple year-over-year % change.
- Population/GDP from the World Bank, R&D spend from OECD - fetched live (see
  `Funding_analysis_helpers.py`).

In [48]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from Funding_analysis_helpers import (
    split_multi_value, fetch_worldbank_indicator, fetch_oecd_gerd, fetch_usd_to_eur_rate,
    lookup_country_value, COUNTRY_NAME_TO_OECD, yoy_growth, YEAR_COLS, COI
)

FINAL_FILE = Path('data_output/20260930_funding_data_cleaned.csv')
OUTPUT_DIR = Path('data_output/analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FUNDING_COL = 'Gov & NP contribution (EUR)'
YEAR_COL_STRS = [str(y) for y in YEAR_COLS]
ALL_TIME_YEARS = [str(y) for y in range(2020, 2026)]  # 2020-2025, matching the template's "All time" window

# Tied to FINAL_FILE's own run-timestamp prefix, not "now" - change FINAL_FILE above to analyze a
# different run and this follows automatically. Reruns of this notebook overwrite the same CSVs
# instead of piling up a fresh timestamped copy every time.
#RUN_TIMESTAMP = re.match(r'(\d{6}_\d{4})', FINAL_FILE.name).group(1)


def save(df_or_series, name, **to_csv_kwargs):
    path = OUTPUT_DIR / f'{name}.csv'
    df_or_series.to_csv(path, **to_csv_kwargs)
    print(f"Saved {name} -> {path}")

### Load and filter

Loads the curated dataset and applies the pipeline-wide filtering rules below - every analysis in
this notebook operates on the filtered `df`, not the raw `FINAL_FILE` export directly. Three filters:

1. Drop grants whose `'Award purpose'` is *only* `'Equipment and infrastructure'`.
2. Drop grants whose `'Funding decision'` is not `'Awarded'` (pledged, not yet decided, unfunded, or blank).
3. Drop grants whose `'Year project started'` is outside 2020-2025 - the reporting window every
   analysis below is labelled with.


In [49]:
df = pd.read_csv(FINAL_FILE)
print(f"Loaded {len(df)} rows from '{FINAL_FILE.name}'.")

# Filter 1: exclude grants whose Award purpose is ONLY 'Equipment and infrastructure' - a grant
# combining it with something else (e.g. 'Research and development; Equipment and infrastructure')
# stays in. Confirmed against the colleagues' own 'Europe wide' template, which applies the same rule.
award_purpose_mask = df['Award purpose'].astype(str).str.strip() == 'Equipment and infrastructure'
df = df[~award_purpose_mask].reset_index(drop=True)
print(f"Excluded {award_purpose_mask.sum()} grants whose Award purpose is ONLY 'Equipment and infrastructure'.")

# Filter 2: keep ONLY grants whose Funding decision is 'Awarded' - money that's been pledged
# ('Committed'), not yet decided ('No decision yet'), turned down ('Unfunded'), or simply unknown
# (blank) shouldn't count in a funding-received analysis. Switched from excluding just 'Committed'
# to this whitelist since several blank/'Unfunded'/'No decision yet' grants turned out to carry
# real funding amounts too (~€15M combined) - not just Committed. This also still removes the
# €162.5M Danish government commitment the colleagues' template excludes from every total
# (confirmed its own Funding decision is 'Committed').
awarded_mask = df['Funding decision'].astype(str).str.strip() == 'Awarded'
df = df[awarded_mask].reset_index(drop=True)
print(f"Excluded {(~awarded_mask).sum()} grants whose Funding decision is not 'Awarded' "
      f"(Committed/Unfunded/No decision yet/blank).")

# Filter 3: restrict to the 2020-2025 reporting window. Applied here, once, rather than at each
# aggregation - previously only SOME cells filtered on year, so grants outside the window leaked
# into totals whose titles said "2020-2025" (1c, 3a, 3b, 4, 4b, 7 and 9). Doing it globally is
# what keeps every analysis in this notebook comparable to every other.
# Two consequences worth knowing:
#  - 1b (funding by FUNDING year) still spreads over 2020-2035, but now only for grants that
#    STARTED in the window. That's the intended reading: "grants started 2020-2025, paid out over time".
#  - Analysis 5's 'Total' column is now window-scoped too (see the note in that cell).
# .between() also drops blank years; there are currently none, so the count below is the whole story.
year_mask = df['Year project started'].between(2020, 2025)
df = df[year_mask].reset_index(drop=True)
print(f"Excluded {(~year_mask).sum()} grants whose start year falls outside 2020-2025.")

# Curation correction: 3 European Commission grants carry Funder Country 'Belgium' (the EC's seat in
# Brussels) instead of 'EU'. Uncorrected they put EUR 12,245,242 of EU-level money into Belgium's
# total - 77% of Belgium's entire figure - and make the by-funder-name and by-country views of EU
# funding disagree by that amount (464.5M vs 452.8M). Funder region is already 'Europe' on all three,
# so only the country needs changing. Targeted on purpose: it only fires on rows that are exactly
# 'European Commission' + 'Belgium', so it becomes a no-op once the source data is fixed.
_ec_belgium = ((df['Funder name'].astype(str).str.strip() == 'European Commission') &
               (df['Funder Country'].astype(str).str.strip() == 'Belgium'))
if _ec_belgium.any():
    print(f"Recoded {_ec_belgium.sum()} European Commission grants from Funder Country 'Belgium' to "
          f"'EU' (EUR {df.loc[_ec_belgium, FUNDING_COL].sum():,.0f}).")
    df.loc[_ec_belgium, 'Funder Country'] = 'EU'

print(f"{len(df)} grants remain for analysis.")


Loaded 2520 rows from '20260930_funding_data_cleaned.csv'.
Excluded 27 grants whose Award purpose is ONLY 'Equipment and infrastructure'.
Excluded 33 grants whose Funding decision is not 'Awarded' (Committed/Unfunded/No decision yet/blank).
Excluded 4 grants whose start year falls outside 2020-2025.
2456 grants remain for analysis.


In [50]:
# Excluding grants from non-profit/NGO funders.
# NOTE: distinct variable names (*_np) on purpose
NON_PROFIT_FUNDER_TYPES = {'nonprofit', 'ngo'}  # matched after lowercasing and stripping -/spaces

funder_type_norm = (df['Funder type'].astype(str)
                    .str.strip().str.lower()
                    .str.replace(r'[-\s]', '', regex=True))
non_profit_mask = funder_type_norm.isin(NON_PROFIT_FUNDER_TYPES)

df_np = df[~non_profit_mask].reset_index(drop=True)
print(f"Excluded {non_profit_mask.sum()} grants with a non-profit/NGO Funder type "
      f"(EUR {df.loc[non_profit_mask, FUNDING_COL].sum():,.0f}). {len(df_np)} grants remain.")

Excluded 213 grants with a non-profit/NGO Funder type (EUR 248,673,492). 2243 grants remain.


### 1. Total funding by start year, and by funding year

In [51]:
by_start_year = df[df['Year project started'].between(2020, 2025)].groupby('Year project started')[FUNDING_COL].sum()
by_start_year.index = by_start_year.index.astype(int)
by_start_year.name = 'Total funding (EUR)'
save(by_start_year, '1a_funding_by_start_year')
print(by_start_year)

by_funding_year = df[YEAR_COL_STRS].sum()
by_funding_year.name = 'Total funding (EUR)'
save(by_funding_year, '1b_funding_by_funding_year')
print()
print(by_funding_year)

Saved 1a_funding_by_start_year -> data_output\analysis\1a_funding_by_start_year.csv
Year project started
2020    1.351691e+08
2021    2.385229e+08
2022    3.021175e+08
2023    4.415752e+08
2024    5.314605e+08
2025    3.116437e+08
Name: Total funding (EUR), dtype: float64
Saved 1b_funding_by_funding_year -> data_output\analysis\1b_funding_by_funding_year.csv

2020    900000750000700000535302.53395647.333311476701...
2021    11170801038142.857480240649371.93751040170.261...
2022    4333333.33311170801038142.857480240649371.9375...
2023    4333333.3334333333.33311170801038142.857480240...
2024    4333333.3334333333.33311170801038142.857317360...
2025                                     342240823.437231
2026    4333333.3331038142.8573173601.4835000001829938...
2027                                     177413772.414257
2028    3173601.481829938.714202136.22722505923.618138...
2029                                      69382592.663147
2030                                      18348727.346157


### 1c. Total funding by region (2020-2025), plus an EU / EU+ / Europe comparison

The second table isn't a partition like the first - EU and EU+ are subsets of Europe, so their
totals are nested (EU <= EU+ <= Europe), not additive alongside the other regions.

In [52]:
from Funding_analysis_helpers import europe_tier_split

# Uses the Totals column directly (not the 2020-2035 per-year split) - many grants lack enough
# public info to know their duration, which would otherwise silently drop them from a "total".
region_split = split_multi_value(df, 'Funder region', delimiter=',')
region_split['weighted_funding'] = region_split[FUNDING_COL] * region_split['_weight']
by_region = region_split.groupby('Funder region_split')['weighted_funding'].sum().sort_values(ascending=False)
by_region.name = 'Total funding (2020-2025, EUR)'
save(by_region, '1c_funding_by_region')
print(by_region)

tier_split = europe_tier_split(df)
tier_split['weighted_funding'] = tier_split[FUNDING_COL] * tier_split['_weight']
by_tier = tier_split.groupby('_tiers')['weighted_funding'].sum().reindex(['EU', 'EU+', 'Europe'])
by_tier.name = 'Total funding (2020-2025, EUR)'
save(by_tier, '1c_funding_by_eu_tier')
print()
print(by_tier)

Saved 1c_funding_by_region -> data_output\analysis\1c_funding_by_region.csv
Funder region_split
Europe               1.272965e+09
North America        3.721569e+08
Asia/Asia Pacific    2.813736e+08
Middle East          2.745611e+07
Global               3.338193e+06
South America        2.639045e+06
Africa               5.599000e+05
Name: Total funding (2020-2025, EUR), dtype: float64
Saved 1c_funding_by_eu_tier -> data_output\analysis\1c_funding_by_eu_tier.csv

_tiers
EU        1.038998e+09
EU+       1.272815e+09
Europe    1.272965e+09
Name: Total funding (2020-2025, EUR), dtype: float64


### 2. Total funding by year x AP pillar, plus growth rates

In [53]:
# By START year (2020-2025), not the calculated per-year split stretching out to 2035 - each
# grant's full funding figure counts entirely toward the year it started, matching analysis 1a's
# methodology exactly but split by AP pillar.
by_start_year_pillar = df[df['Year project started'].between(2020, 2025)].pivot_table(
    index='AP pillar', columns='Year project started', values=FUNDING_COL, aggfunc='sum', fill_value=0
)
by_start_year_pillar.columns = by_start_year_pillar.columns.astype(int)
by_start_year_pillar = by_start_year_pillar[sorted(by_start_year_pillar.columns)]
by_start_year_pillar.loc['Total'] = by_start_year_pillar.sum()
save(by_start_year_pillar, '2a_funding_by_year_and_pillar')
print(by_start_year_pillar)

# YoY growth rate (matches the template's own formula: (year_N - year_N-1) / year_N-1)
growth_rates = by_start_year_pillar.apply(yoy_growth, axis=1)
growth_rates['Average growth rate (2021-2025)'] = growth_rates.iloc[:, 1:].mean(axis=1)
save(growth_rates, '2b_growth_rates_by_pillar')
print()
print(growth_rates)

Saved 2a_funding_by_year_and_pillar -> data_output\analysis\2a_funding_by_year_and_pillar.csv
Year project started          2020          2021          2022          2023  \
AP pillar                                                                      
Cross-cutting         1.082863e+07  5.153889e+07  5.808985e+07  9.767319e+07   
Cultivated            1.646469e+07  4.050796e+07  4.915279e+07  6.472893e+07   
Fermentation          2.175219e+07  4.266925e+07  4.716501e+07  1.166055e+08   
Plant-based           8.612357e+07  1.038068e+08  1.477098e+08  1.625676e+08   
Total                 1.351691e+08  2.385229e+08  3.021175e+08  4.415752e+08   

Year project started          2024          2025  
AP pillar                                         
Cross-cutting         1.731158e+08  3.267131e+07  
Cultivated            7.069326e+07  2.978128e+07  
Fermentation          1.665243e+08  1.456873e+08  
Plant-based           1.211271e+08  1.035038e+08  
Total                 5.314605e+08  3.1

### 2c/2d. Funding by year x region, and by EU / EU+ / Europe, with growth rates

In [54]:
# By START year (2020-2025), not the funding-year spread - matches 1a/2a's methodology,
# split by region. 1b is the one deliberate exception that stays funding-year-based.
by_year_region = region_split[region_split['Year project started'].between(2020, 2025)].pivot_table(
    index='Funder region_split', columns='Year project started', values='weighted_funding', aggfunc='sum', fill_value=0
)
by_year_region.columns = by_year_region.columns.astype(int)
by_year_region = by_year_region[sorted(by_year_region.columns)]
by_year_region.loc['Total'] = by_year_region.sum()
save(by_year_region, '2c_funding_by_year_and_region')
print(by_year_region)

growth_region = by_year_region.apply(yoy_growth, axis=1)
growth_region['Average growth rate (2021-2025)'] = growth_region.iloc[:, 1:].mean(axis=1)
save(growth_region, '2d_growth_rates_by_region')
print()
print(growth_region)

Saved 2c_funding_by_year_and_region -> data_output\analysis\2c_funding_by_year_and_region.csv
Year project started          2020          2021          2022          2023  \
Funder region_split                                                            
Africa                0.000000e+00  0.000000e+00  0.000000e+00  5.599000e+05   
Asia/Asia Pacific     1.304494e+07  7.278290e+07  3.753042e+07  2.870742e+07   
Europe                8.792323e+07  1.213803e+08  2.083944e+08  3.395904e+08   
Global                1.309406e+06  6.024936e+05  8.956450e+05  5.306482e+05   
Middle East           0.000000e+00  8.149300e+06  1.666206e+07  2.271742e+06   
North America         3.288723e+07  3.559801e+07  3.859720e+07  6.821811e+07   
South America         4.269000e+03  9.929000e+03  3.773600e+04  1.696999e+06   
Total                 1.351691e+08  2.385229e+08  3.021175e+08  4.415752e+08   

Year project started          2024          2025  
Funder region_split                               
Afr

In [55]:
by_year_tier = tier_split[tier_split['Year project started'].between(2020, 2025)].pivot_table(
    index='_tiers', columns='Year project started', values='weighted_funding', aggfunc='sum', fill_value=0
).reindex(['EU', 'EU+', 'Europe'])
by_year_tier.columns = by_year_tier.columns.astype(int)
by_year_tier = by_year_tier[sorted(by_year_tier.columns)]
save(by_year_tier, '2c_funding_by_year_and_eu_tier')
print(by_year_tier)

growth_tier = by_year_tier.apply(yoy_growth, axis=1)
growth_tier['Average growth rate (2021-2025)'] = growth_tier.iloc[:, 1:].mean(axis=1)
save(growth_tier, '2d_growth_rates_by_eu_tier')
print()
print(growth_tier)

Saved 2c_funding_by_year_and_eu_tier -> data_output\analysis\2c_funding_by_year_and_eu_tier.csv
Year project started          2020          2021          2022          2023  \
_tiers                                                                         
EU                    7.272313e+07  7.502605e+07  1.842130e+08  2.814545e+08   
EU+                   8.792323e+07  1.213803e+08  2.083944e+08  3.394404e+08   
Europe                8.792323e+07  1.213803e+08  2.083944e+08  3.395904e+08   

Year project started          2024          2025  
_tiers                                            
EU                    2.199877e+08  2.055937e+08  
EU+                   2.892173e+08  2.264595e+08  
Europe                2.892173e+08  2.264595e+08  
Saved 2d_growth_rates_by_eu_tier -> data_output\analysis\2d_growth_rates_by_eu_tier.csv

Year project started  2020      2021      2022      2023      2024      2025  \
_tiers                                                                         

### 3a. Total funding by country (2020-2025), with per-capita/GDP/R&D-spend

Fetches World Bank population/GDP and OECD R&D spend live, plus a live USD->EUR rate (Frankfurter)
to convert them - reused by analyses 4 and 8 below, so this cell's variables stay in scope.

In [56]:
# No year filter needed here - `df` is already restricted to 2020-2025 by Filter 3 in the load
# cell. `country_split` is reused by analyses 4, 6 and 9 below, so it inherits that window too.
country_split = split_multi_value(df, 'Funder Country', delimiter=',')
country_split['weighted_funding'] = country_split[FUNDING_COL] * country_split['_weight']

by_country = country_split.groupby('Funder Country_split')['weighted_funding'].sum().sort_values(ascending=False)
by_country.name = 'Total funding (2020-2025, EUR)'

print("Fetching World Bank population/GDP, OECD R&D spend, and a USD->EUR rate...")
POP_GDP_YEAR = 2023  # most recent year with broad World Bank/OECD coverage
pop = fetch_worldbank_indicator('SP.POP.TOTL', POP_GDP_YEAR)
gdp_usd = fetch_worldbank_indicator('NY.GDP.MKTP.CD', POP_GDP_YEAR)
gerd_usd = fetch_oecd_gerd(POP_GDP_YEAR)
usd_to_eur = fetch_usd_to_eur_rate(f'{POP_GDP_YEAR}-12-29')
print(f"USD->EUR rate ({POP_GDP_YEAR}-12-29): {usd_to_eur}")

by_country_df = by_country.reset_index().rename(columns={'Funder Country_split': 'Country'})
by_country_df['Population'] = by_country_df['Country'].apply(lambda c: lookup_country_value(c, pop))
by_country_df['GDP (EUR)'] = by_country_df['Country'].apply(lambda c: lookup_country_value(c, gdp_usd)) * usd_to_eur
by_country_df['R&D spend (EUR)'] = by_country_df['Country'].apply(
    lambda c: lookup_country_value(c, gerd_usd, COUNTRY_NAME_TO_OECD)
) * usd_to_eur
by_country_df['Funding per capita'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['Population']
by_country_df['Funding per GDP'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['GDP (EUR)']
by_country_df['GDP per capita'] = by_country_df['GDP (EUR)'] / by_country_df['Population']
by_country_df['Funding per R&D spend'] = by_country_df['Total funding (2020-2025, EUR)'] / by_country_df['R&D spend (EUR)']

n_no_population = by_country_df['Population'].isna().sum()
print(f"{n_no_population} entries have no population match (expected for non-country entities like 'EU'/'Global').")

save(by_country_df, '3a_funding_by_country', index=False)
print(by_country_df.head(15).to_string())

Fetching World Bank population/GDP, OECD R&D spend, and a USD->EUR rate...


c:\Users\Viola Hollek\Documents\SHEEP_Pipeline\Funding\Funding_analysis_helpers.py:150: DtypeWarning: Columns (0: OBS_STATUS, 1: Observation status, 2: OBS_STATUS_2, 3: Observation status 2, 4: OBS_STATUS_3, 5: Observation status 3, 6: AUX_OBS_STATUS, 7: Aux observation status, 8: CONF_STATUS, 9: Confidentiality status) have mixed types. Specify dtype option on import or set low_memory=False.
  raw = pd.read_csv(io.StringIO(text))


USD->EUR rate (2023-12-29): 0.90498
2 entries have no population match (expected for non-country entities like 'EU'/'Global').
Saved 3a_funding_by_country -> data_output\analysis\3a_funding_by_country.csv
           Country  Total funding (2020-2025, EUR)   Population     GDP (EUR)  R&D spend (EUR)  Funding per capita  Funding per GDP  GDP per capita  Funding per R&D spend
0               EU                    4.544506e+08          NaN           NaN              NaN                 NaN              NaN             NaN                    NaN
1    United States                    3.093985e+08  336755052.0  2.516887e+13     8.616450e+11            0.918764         0.000012    74739.388482               0.000359
2          Denmark                    1.642351e+08    5946952.0  3.662017e+11     1.277912e+10           27.616682         0.000448    61578.048890               0.012852
3   United Kingdom                    1.495972e+08   68526000.0  3.095753e+12     1.037927e+11            2.183

### 3b. Total funding by country (2020-2025) excl. non-profit funding, with per-capita/GDP/R&D-spend

Same breakdown as 3a with grants from non-profit/NGO funders removed, i.e. `'Funder type'` not in
`'Non-profit'` / `'Nonprofit'` / `'NGO'`. Compared case/hyphen/whitespace-insensitively because the
curated data carries both `'Non-profit'` and `'Nonprofit'` as separate labels for the same thing.

`'Non-profit/Gov'` (4 grants) is deliberately **kept** - mixed public/non-profit funders count as
partly governmental. `'For profit'`, `'Research Institute'`, `'Joint public/private partnership'`,
`'Unknown'` and blank Funder types are also kept; this filter only removes purely non-profit/NGO money.


In [57]:
# Same breakdown as 3a, but excluding grants from non-profit/NGO funders using the dataframe df_np

country_split_np = split_multi_value(df_np, 'Funder Country', delimiter=',')
country_split_np['weighted_funding'] = country_split_np[FUNDING_COL] * country_split_np['_weight']

by_country_np = (country_split_np.groupby('Funder Country_split')['weighted_funding']
                 .sum().sort_values(ascending=False))
# Same column header as 3a so this CSV pastes into the same template columns.
by_country_np.name = 'Total funding (2020-2025, EUR)'

# pop / gdp_usd / gerd_usd / usd_to_eur were fetched in 3a and are still in scope - deliberately
# not refetched here (4 extra network round-trips that can fail). Run 3a before 3b.
by_country_np_df = by_country_np.reset_index().rename(columns={'Funder Country_split': 'Country'})
by_country_np_df['Population'] = by_country_np_df['Country'].apply(lambda c: lookup_country_value(c, pop))
by_country_np_df['GDP (EUR)'] = by_country_np_df['Country'].apply(lambda c: lookup_country_value(c, gdp_usd)) * usd_to_eur
by_country_np_df['R&D spend (EUR)'] = by_country_np_df['Country'].apply(
    lambda c: lookup_country_value(c, gerd_usd, COUNTRY_NAME_TO_OECD)
) * usd_to_eur
by_country_np_df['Funding per capita'] = by_country_np_df['Total funding (2020-2025, EUR)'] / by_country_np_df['Population']
by_country_np_df['Funding per GDP'] = by_country_np_df['Total funding (2020-2025, EUR)'] / by_country_np_df['GDP (EUR)']
by_country_np_df['GDP per capita'] = by_country_np_df['GDP (EUR)'] / by_country_np_df['Population']
by_country_np_df['Funding per R&D spend'] = by_country_np_df['Total funding (2020-2025, EUR)'] / by_country_np_df['R&D spend (EUR)']

n_no_population = by_country_np_df['Population'].isna().sum()
print(f"{n_no_population} entries have no population match (expected for non-country entities like 'EU'/'Global').")

save(by_country_np_df, '3b_funding_by_country_excl_nonprofit', index=False)
print(by_country_np_df.head(15).to_string())


1 entries have no population match (expected for non-country entities like 'EU'/'Global').
Saved 3b_funding_by_country_excl_nonprofit -> data_output\analysis\3b_funding_by_country_excl_nonprofit.csv
           Country  Total funding (2020-2025, EUR)   Population     GDP (EUR)  R&D spend (EUR)  Funding per capita  Funding per GDP  GDP per capita  Funding per R&D spend
0               EU                    4.544506e+08          NaN           NaN              NaN                 NaN              NaN             NaN                    NaN
1    United States                    1.910789e+08  336755052.0  2.516887e+13     8.616450e+11            0.567412         0.000008    74739.388482               0.000222
2   United Kingdom                    1.480611e+08   68526000.0  3.095753e+12     1.037927e+11            2.160656         0.000048    45176.320750               0.001427
3      South Korea                    1.411758e+08   51712619.0  1.669508e+12     1.328646e+11            2.730007   

### 4. Total funding by country x AP pillar (2020-2025)

In [58]:
country_pillar = country_split.pivot_table(
    index='Funder Country_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
country_pillar['Total'] = country_pillar.sum(axis=1)
country_pillar = country_pillar.sort_values('Total', ascending=False)
save(country_pillar, '4_funding_by_country_and_pillar')
print(country_pillar)

Saved 4_funding_by_country_and_pillar -> data_output\analysis\4_funding_by_country_and_pillar.csv
AP pillar             Cross-cutting    Cultivated  Fermentation   Plant-based  \
Funder Country_split                                                            
EU                     3.877506e+07  2.641055e+07  1.778421e+08  2.114229e+08   
United States          9.470778e+07  3.277099e+07  1.341277e+08  4.779206e+07   
Denmark                6.085037e+06  2.890231e+06  6.443271e+07  9.082711e+07   
United Kingdom         4.117117e+07  2.742822e+07  3.991223e+07  4.108559e+07   
South Korea            6.666517e+07  6.371271e+07  5.987246e+06  4.810685e+06   
Netherlands            6.164645e+07  1.452487e+07  2.305151e+06  1.640810e+07   
Singapore              1.285894e+07  1.798270e+07  3.960247e+07  1.269518e+07   
Germany                4.181583e+06  9.593932e+06  7.620307e+06  6.058107e+07   
Finland                3.864689e+07  4.899000e+05  2.682890e+07  1.179841e+07   
Canada     

### 4a. Total funding by country x AP pillar excl. non-profit funding

Same breakdown as 4, restricted to the same non-profit/NGO-excluded set as 3b (see 3b's note for
exactly which `'Funder type'` values are dropped and which are kept). Reuses 3b's
`country_split_np`, so **3b must have been run first**.


In [59]:
# Same pivot as 4, but off 3b's non-profit-excluded split rather than re-deriving the filter -
# guarantees 4a and 3b always agree on which grants are in scope. `country_split_np` already
# carries the weighted_funding column, so this is a pure re-pivot.
# NOTE: *_np naming again on purpose - analyses 6 and 9 below read 3a's `country_split`.
country_pillar_np = country_split_np.pivot_table(
    index='Funder Country_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
country_pillar_np['Total'] = country_pillar_np.sum(axis=1)
country_pillar_np = country_pillar_np.sort_values('Total', ascending=False)

# Reindex onto 4's country list so 4a has the SAME rows in the same order-ish as 4, making the two
# CSVs directly comparable side by side in the template. A country whose funding was entirely
# non-profit drops out of country_split_np and would otherwise vanish from this table; showing it
# as an explicit 0 row says "assessed, nothing left" rather than silently omitting it.
# In the current data that applies to 'Global' only. ('Chile', 'Iceland', 'Malaysia', 'Italy',
# 'Russia' and 'Turkey' also read 0 here, but they are already 0 in 4 - grants with no funding
# amount recorded, not an artefact of this filter.)
country_pillar_np = country_pillar_np.reindex(
    country_pillar.index.union(country_pillar_np.index, sort=False), fill_value=0
).fillna(0).sort_values('Total', ascending=False)

save(country_pillar_np, '4a_funding_by_country_and_pillar_excl_nonprofit')
print(country_pillar_np)


Saved 4a_funding_by_country_and_pillar_excl_nonprofit -> data_output\analysis\4a_funding_by_country_and_pillar_excl_nonprofit.csv
AP pillar             Cross-cutting    Cultivated  Fermentation   Plant-based  \
Funder Country_split                                                            
EU                     3.877506e+07  2.641055e+07  1.778421e+08  2.114229e+08   
United States          1.375888e+07  2.467003e+07  1.170682e+08  3.558181e+07   
United Kingdom         4.117117e+07  2.742822e+07  3.961206e+07  3.984964e+07   
South Korea            6.666517e+07  6.371271e+07  5.987246e+06  4.810685e+06   
Netherlands            6.164645e+07  1.452487e+07  2.305151e+06  1.640810e+07   
Singapore              1.285894e+07  1.798270e+07  3.960247e+07  1.269518e+07   
Germany                4.181583e+06  9.593932e+06  7.620307e+06  6.058107e+07   
Finland                3.858689e+07  4.899000e+05  2.682890e+07  1.179841e+07   
Norway                 1.689705e+07  1.606977e+06  1.082230e

### 4b. Total funding by region x AP pillar (2020-2025), and by EU / EU+ / Europe x AP pillar

In [60]:
region_pillar = region_split.pivot_table(
    index='Funder region_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
region_pillar['Total'] = region_pillar.sum(axis=1)
region_pillar = region_pillar.sort_values('Total', ascending=False)
save(region_pillar, '4b_funding_by_region_and_pillar')
print(region_pillar)

tier_pillar = tier_split.pivot_table(
    index='_tiers', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
).reindex(['EU', 'EU+', 'Europe'])
tier_pillar['Total'] = tier_pillar.sum(axis=1)
save(tier_pillar, '4b_funding_by_eu_tier_and_pillar')
print()
print(tier_pillar)

Saved 4b_funding_by_region_and_pillar -> data_output\analysis\4b_funding_by_region_and_pillar.csv


AP pillar            Cross-cutting    Cultivated  Fermentation   Plant-based  \
Funder region_split                                                            
Europe                2.254572e+08  1.071217e+08  3.513914e+08  5.889948e+08   
North America         9.504176e+07  4.192354e+07  1.355070e+08  9.968469e+07   
Asia/Asia Pacific     9.348106e+07  1.013311e+08  5.172974e+07  3.483169e+07   
Middle East           9.491092e+06  1.750923e+07  4.557900e+05  0.000000e+00   
Global                0.000000e+00  1.880443e+06  6.630000e+05  7.947500e+05   
South America         4.465736e+05  1.562936e+06  9.673755e+04  5.327980e+05   
Africa                0.000000e+00  0.000000e+00  5.599000e+05  0.000000e+00   

AP pillar                   Total  
Funder region_split                
Europe               1.272965e+09  
North America        3.721569e+08  
Asia/Asia Pacific    2.813736e+08  
Middle East          2.745611e+07  
Global               3.338193e+06  
South America        2.6390

### 4c. Total funding by region x AP pillar excl. non-profit funding, and by EU / EU+ / Europe x AP pillar

Region- and tier-level twins of 4a, on the same non-profit/NGO-excluded set as 3b (see 3b's note for
which `'Funder type'` values are dropped). Also defines `region_split_np` / `tier_split_np`, which 9c
reuses - so **3b must have been run first**, and this cell before 9c.


In [61]:
# Region/tier equivalents of 4a. Built from `df_np` (defined once in 3b) rather than re-deriving the
# funder-type filter, so 3b, 4a, 4c and 9c can never disagree about which grants are in scope.
# NOTE: *_np naming again - analyses 5-9 read the unfiltered `region_split` / `tier_split`.
region_split_np = split_multi_value(df_np, 'Funder region', delimiter=',')
region_split_np['weighted_funding'] = region_split_np[FUNDING_COL] * region_split_np['_weight']

tier_split_np = europe_tier_split(df_np)
tier_split_np['weighted_funding'] = tier_split_np[FUNDING_COL] * tier_split_np['_weight']

region_pillar_np = region_split_np.pivot_table(
    index='Funder region_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
region_pillar_np['Total'] = region_pillar_np.sum(axis=1)
# Reindexed onto 4b's region list for the same reason 4a is reindexed onto 4's - keeps the two CSVs
# row-for-row comparable in the template. 'Global' funding is 100% non-profit, so it appears here as
# an explicit 0 row rather than silently vanishing.
region_pillar_np = region_pillar_np.reindex(
    region_pillar.index.union(region_pillar_np.index, sort=False), fill_value=0
).fillna(0).sort_values('Total', ascending=False)
save(region_pillar_np, '4c_funding_by_region_and_pillar_excl_nonprofit')
print(region_pillar_np)

tier_pillar_np = tier_split_np.pivot_table(
    index='_tiers', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
).reindex(['EU', 'EU+', 'Europe'])
tier_pillar_np['Total'] = tier_pillar_np.sum(axis=1)
save(tier_pillar_np, '4c_funding_by_eu_tier_and_pillar_excl_nonprofit')
print()
print(tier_pillar_np)


Saved 4c_funding_by_region_and_pillar_excl_nonprofit -> data_output\analysis\4c_funding_by_region_and_pillar_excl_nonprofit.csv
AP pillar            Cross-cutting    Cultivated  Fermentation   Plant-based  \
Funder region_split                                                            
Europe                2.221305e+08  1.048306e+08  2.944108e+08  5.284485e+08   
Asia/Asia Pacific     9.348106e+07  1.013004e+08  5.172974e+07  3.483169e+07   
North America         1.409286e+07  3.321859e+07  1.184474e+08  8.461114e+07   
Middle East           9.491092e+06  1.713622e+07  4.557900e+05  0.000000e+00   
South America         4.465736e+05  1.562936e+06  9.673755e+04  5.327980e+05   
Africa                0.000000e+00  0.000000e+00  5.599000e+05  0.000000e+00   
Global                0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   

AP pillar                   Total  
Funder region_split                
Europe               1.149820e+09  
Asia/Asia Pacific    2.813429e+08  
North A

### 5. Top 20 funder institutions, with yearly split (no pillar split)

In [62]:
# Yearly breakdown by START year, not the funding-year spread - same default as everywhere
# else except 1b. 'Total' is the funder's sum across the 2020-2025 window: since Filter 3 now
# scopes `df` to that window globally, this Total is window-scoped too (it previously spanned
# all start years). Only NSERC was affected in the current data: 137,862 -> 96,985 EUR.
by_funder_year = df[df['Year project started'].between(2020, 2025)].pivot_table(
    index='Funder name', columns='Year project started', values=FUNDING_COL, aggfunc='sum', fill_value=0
)
by_funder_year.columns = by_funder_year.columns.astype(int)
by_funder_year = by_funder_year[sorted(by_funder_year.columns)]
by_funder_year['Total'] = df.groupby('Funder name')[FUNDING_COL].sum()
by_funder_year = by_funder_year.fillna(0)
top20_funders = by_funder_year.sort_values('Total', ascending=False).head(20)
save(top20_funders, '5_top20_funders')
print(top20_funders)

Saved 5_top20_funders -> data_output\analysis\5_top20_funders.csv
Year project started                                        2020  \
Funder name                                                        
European Commission                                 3.468856e+07   
UK Research and Innovation (UKRI)                   2.992053e+06   
Novo Nordisk Foundation                             3.345500e+04   
Bezos Earth Fund (BEF)                              0.000000e+00   
Business Finland                                    3.272000e+06   
Dutch National Growth Fund                          0.000000e+00   
Department of Commerce (DoC)                        0.000000e+00   
Research Council of Norway                          1.022493e+07   
"Agency for Science, Technology, and Research (...  0.000000e+00   
Protein Industries Canada                           1.955580e+07   
Federal Ministry of Food and Agriculture (BMEL)     1.552889e+06   
Defense Advanced Research Projects Agency (DARPA) 

### 6. Top 20 country rankings per year, 2020-2025 (non-cumulative)

Two shapes: (a) rank 1-20 x year, country name per cell (for a table); (b) country x year, rank
number per cell (for graphing a ranked-line/bump chart). Distinct from analysis #8's cumulative
ranking below - confirmed via the template these are two different things, not variants of one.

In [63]:
# By START year (2020-2025), not the funding-year spread - same default as everywhere else in
# this notebook except 1b.
annual_country = country_split[country_split['Year project started'].between(2020, 2025)]
year_country_totals = {}
for y in range(2020, 2026):
    year_country_totals[y] = annual_country[annual_country['Year project started'] == y].groupby(
        'Funder Country_split'
    )['weighted_funding'].sum().sort_values(ascending=False)

# Shape A: rank (1-20) x year, country name per cell
rank_table = pd.DataFrame({y: year_country_totals[y].head(20).index.tolist() for y in range(2020, 2026)})
rank_table.index = pd.RangeIndex(1, 21, name='Rank')
save(rank_table, '6a_country_rank_table')
print(rank_table)

# Shape B: country x year, rank number per cell (for graphing)
all_countries_ranked = sorted(set().union(*[year_country_totals[y].head(10).index for y in range(2020, 2026)]))
country_rank_grid = pd.DataFrame(index=all_countries_ranked, columns=range(2020, 2026), dtype='float')
for y in range(2020, 2026):
    ranks = {c: i + 1 for i, c in enumerate(year_country_totals[y].index)}
    for c in all_countries_ranked:
        country_rank_grid.loc[c, y] = ranks.get(c, np.nan)
save(country_rank_grid, '6b_country_rank_grid')
print()
print(country_rank_grid)

Saved 6a_country_rank_table -> data_output\analysis\6a_country_rank_table.csv
                2020            2021            2022            2023  \
Rank                                                                   
1                 EU       Singapore              EU              EU   
2             Canada          Norway         Denmark     Netherlands   
3             Norway   United States     South Korea   United States   
4            Germany              EU   United States  United Kingdom   
5             Sweden         Denmark  United Kingdom         Germany   
6      United States     South Korea          Israel         Denmark   
7        New Zealand           Spain         Germany          France   
8            Denmark          Canada           Spain          Canada   
9            Belgium          Israel         Finland           Japan   
10           Finland  United Kingdom     Switzerland           Spain   
11    United Kingdom          Sweden          Sweden      

### 7. Total funding 2020-2025 by AP pillar x End product type, plus a separate Dairy sub-split

In [64]:
ep_split = split_multi_value(df, 'End product type', delimiter=';')
ep_split['weighted_funding'] = ep_split[FUNDING_COL] * ep_split['_weight']
pillar_end_product = ep_split.pivot_table(
    index='End product type_split', columns='AP pillar', values='weighted_funding', aggfunc='sum', fill_value=0
)
pillar_end_product['Total'] = pillar_end_product.sum(axis=1)
save(pillar_end_product, '7a_funding_by_pillar_and_end_product')
print(pillar_end_product)

# Dairy's own constituents, not cross-tabulated with pillar. Starts from ep_split's Dairy share
# (not df directly) so a 'Meat; Dairy' grant contributes the same weighted share here as it does
# to 7a's Dairy total, then splits that share further across sub-end products. 'sub-end product'
# is exclusively populated for Dairy rows (confirmed: no non-Dairy row has one) - Dairy rows
# missing a sub-end product are labelled 'Agnostic' here, for this breakdown only, so their
# funding isn't silently dropped. Not written back to the underlying data: 'Agnostic' is already a
# distinct End product type value, and reusing it as a sub-end product value there would be confusing.
dairy_ep = ep_split[ep_split['End product type_split'] == 'Dairy'].copy()
dairy_ep['sub-end product'] = dairy_ep['sub-end product'].fillna('Agnostic')
dairy_ep = dairy_ep.rename(columns={'weighted_funding': 'dairy_weighted_funding'})
sub_ep_split = split_multi_value(dairy_ep, 'sub-end product', delimiter=';')
sub_ep_split['weighted_funding'] = sub_ep_split['dairy_weighted_funding'] * sub_ep_split['_weight']
dairy_breakdown = sub_ep_split.groupby('sub-end product_split')['weighted_funding'].sum().sort_values(ascending=False)
dairy_breakdown.name = 'Total funding (2020-2025, EUR)'
save(dairy_breakdown, '7b_dairy_sub_breakdown')
print()
print(dairy_breakdown)

Saved 7a_funding_by_pillar_and_end_product -> data_output\analysis\7a_funding_by_pillar_and_end_product.csv
AP pillar                               Cross-cutting    Cultivated  \
End product type_split                                                
Agnostic                                 3.201466e+08  5.267538e+07   
Chocolate, desserts, and confectionery   0.000000e+00  3.597543e+05   
Dairy                                    5.628772e+06  1.185138e+06   
Eggs and egg proteins                    5.498261e+06  0.000000e+00   
Fish and seafood                         2.018002e+07  2.719244e+07   
Meat                                     7.246404e+07  1.897692e+08   
Spreads, sauces, and condiments          0.000000e+00  0.000000e+00   

AP pillar                               Fermentation   Plant-based  \
End product type_split                                               
Agnostic                                4.536600e+08  5.169338e+08   
Chocolate, desserts, and confectionery  1.

### 7c/7d. End product type by country, and the Dairy sub-split by country (2020-2025)

Country columns restricted to `COI` from `Funding_analysis_helpers` (13 funders of interest).

Funding is split **twice**: once across a grant's end product types, then again across its funder
countries - so a `'Meat; Dairy'` grant funded by `'Sweden, EU'` puts 1/4 of its amount in each of
the four cells. Same nesting the Dairy sub-split in 7 already uses.

Reuses `ep_split` and `sub_ep_split` from 7, so **run 7 first**. Row totals cover the COI columns
only and are therefore lower than 7a/7b's world totals - the coverage line reports by how much.


In [65]:
from Funding_analysis_helpers import COI

# 'EU' is treated as a country here, on purpose: it is its own 'Funder Country' value
# (supranational EU-level funding) and the single largest funder in the data, so it needs its own
# column. Unioned in explicitly rather than trusting COI alone, because a Jupyter kernel caches
# Funding_analysis_helpers - a COI edited after the module was first imported stays stale until the
# kernel restarts, which is exactly how the EU column went missing on the first run of this cell.
COUNTRIES_OF_INTEREST = set(COI) | {'EU'}
if 'EU' not in COI:
    print("Note: the imported COI has no 'EU' (stale cached module?) - added it here. "
          "Restart the kernel if you expect other COI edits to be picked up.")


def _by_country(base_split, value_col, index_col):
    """Split an already-end-product-weighted frame across funder countries and pivot to
    index_col x country, keeping only COI columns. Renames the incoming weight column first because
    split_multi_value overwrites '_weight' - the new weight must multiply the OLD share, not replace it."""
    base = base_split.rename(columns={value_col: '_carried'}).copy()
    exploded = split_multi_value(base, 'Funder Country', delimiter=',')
    exploded['weighted_funding'] = exploded['_carried'] * exploded['_weight']
    table = exploded.pivot_table(
        index=index_col, columns='Funder Country_split', values='weighted_funding',
        aggfunc='sum', fill_value=0,
    )
    # Reindex onto the full COI list so the column set is stable even if a country funded nothing.
    table = table.reindex(columns=sorted(COUNTRIES_OF_INTEREST), fill_value=0).fillna(0)
    table['Total (COI)'] = table.sum(axis=1)
    return table.sort_values('Total (COI)', ascending=False), exploded


# 7c: end product type x country
end_product_country, _ep_country_exploded = _by_country(
    ep_split, 'weighted_funding', 'End product type_split')
save(end_product_country, '7c_funding_by_end_product_and_country')
print(end_product_country.to_string(float_format=lambda v: f'{v:,.0f}'))

_coi_total = end_product_country['Total (COI)'].sum()
_world_total = _ep_country_exploded['weighted_funding'].sum()
print(f"\nCOI columns cover EUR {_coi_total:,.0f} of EUR {_world_total:,.0f} "
      f"({100 * _coi_total / _world_total:.1f}%) of end-product-attributed funding.")

# 7d: Dairy sub-end product x country. sub_ep_split already carries the Dairy share split across
# sub-end products, so this only adds the country dimension.
dairy_sub_country, _dairy_country_exploded = _by_country(
    sub_ep_split, 'weighted_funding', 'sub-end product_split')
save(dairy_sub_country, '7d_dairy_sub_breakdown_by_country')
print()
print(dairy_sub_country.to_string(float_format=lambda v: f'{v:,.0f}'))

_coi_dairy = dairy_sub_country['Total (COI)'].sum()
_world_dairy = _dairy_country_exploded['weighted_funding'].sum()
print(f"\nCOI columns cover EUR {_coi_dairy:,.0f} of EUR {_world_dairy:,.0f} "
      f"({100 * _coi_dairy / _world_dairy:.1f}%) of Dairy funding.")


Saved 7c_funding_by_end_product_and_country -> data_output\analysis\7c_funding_by_end_product_and_country.csv
Funder Country_split                       Canada     Denmark          EU    Finland    Germany     Israel      Japan     Norway  Singapore  South Korea     Sweden  United Kingdom  United States   Total (COI)
End product type_split                                                                                                                                                                                          
Agnostic                               36,898,279 127,152,571 342,365,677 66,271,407 49,815,854  9,864,098 16,117,455 28,715,067 68,006,710   15,855,117 34,624,436     101,384,072    257,144,988 1,154,215,731
Meat                                   16,793,923  20,741,230  38,046,270  6,264,900 15,173,276 16,662,060  1,645,839 28,436,206 14,035,169   92,875,390  9,046,537      38,814,428     42,584,673   341,119,900
Dairy                                   5,077,559  11,

### 8. Investment and cumulative investment by country, 2020-2025, with cumulative rank

Reuses `year_country_totals` computed for analysis #6 above.

In [66]:
investment_by_year = pd.DataFrame({y: year_country_totals[y] for y in range(2020, 2026)}).fillna(0)
save(investment_by_year, '8a_investment_by_country_and_year')

cumulative_by_year = investment_by_year.cumsum(axis=1)
save(cumulative_by_year, '8b_cumulative_investment_by_country_and_year')

# Rank 1 = highest cumulative total-to-date that year
cumulative_rank = cumulative_by_year.rank(axis=0, ascending=False, method='min')
save(cumulative_rank, '8c_cumulative_investment_rank')

print(investment_by_year.head(10))
print()
print(cumulative_by_year.head(10))
print()
print(cumulative_rank.head(10))

Saved 8a_investment_by_country_and_year -> data_output\analysis\8a_investment_by_country_and_year.csv
Saved 8b_cumulative_investment_by_country_and_year -> data_output\analysis\8b_cumulative_investment_by_country_and_year.csv
Saved 8c_cumulative_investment_rank -> data_output\analysis\8c_cumulative_investment_rank.csv
                              2020          2021          2022          2023  \
Funder Country_split                                                           
Australia             2.060400e+04  4.828994e+06  1.328364e+06  6.063202e+05   
Austria               0.000000e+00  0.000000e+00  6.654340e+05  3.200000e+06   
Belgium               3.600000e+06  0.000000e+00  0.000000e+00  0.000000e+00   
Brazil                0.000000e+00  0.000000e+00  0.000000e+00  1.473885e+06   
Canada                2.343843e+07  9.612336e+06  5.661477e+06  1.334250e+07   
Chile                 0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00   
China                 2.705197e+06  2.32

### 8d. Investment and cumulative investment by country excl. non-profit funding, 2020-2025

Non-profit/NGO-excluded twin of 8a-8c. It rebuilds the per-year country totals from
`country_split_np` (the weighted country split defined once in 3b) rather than reusing
`year_country_totals`, which is all-funders. Run 3b before this cell.


In [67]:
# Non-profit-excluded twin of 8a-8c. Built from `country_split_np` (defined once in 3b) so the
# weights come from the same frame the non-profit mask was applied to - masks are never reindexed
# across frames here, because 3b calls reset_index(drop=True) and the row labels no longer line up.
annual_country_np = country_split_np[country_split_np['Year project started'].between(2020, 2025)]
year_country_totals_np = {
    y: (annual_country_np[annual_country_np['Year project started'] == y]
        .groupby('Funder Country_split')['weighted_funding'].sum()
        .sort_values(ascending=False))
    for y in range(2020, 2026)
}

investment_by_year_np = pd.DataFrame(
    {y: year_country_totals_np[y] for y in range(2020, 2026)}
).fillna(0)
save(investment_by_year_np, '8d_investment_by_country_and_year_excl_nonprofit')

cumulative_by_year_np = investment_by_year_np.cumsum(axis=1)
save(cumulative_by_year_np, '8d_cumulative_investment_by_country_and_year_excl_nonprofit')

# Rank 1 = highest cumulative total-to-date that year. Countries funded ONLY by non-profits drop
# out of the index entirely, so these ranks are not directly comparable to 8c's row-for-row.
cumulative_rank_np = cumulative_by_year_np.rank(axis=0, ascending=False, method='min')
save(cumulative_rank_np, '8d_cumulative_investment_rank_excl_nonprofit')

# Tripwire: the excl.-non-profit total can never exceed the all-funders total of 8a.
_total_all = investment_by_year.to_numpy().sum()
_total_np = investment_by_year_np.to_numpy().sum()
assert _total_np <= _total_all + 1e-6, f"excl.-non-profit total {_total_np:,.0f} exceeds all-funders {_total_all:,.0f}"
_dropped = sorted(set(investment_by_year.index) - set(investment_by_year_np.index))
print(f"2020-2025 total: EUR {_total_all:,.0f} all funders vs EUR {_total_np:,.0f} excl. non-profit "
      f"({_total_np / _total_all:.1%}).")
print(f"{len(_dropped)} countries drop out entirely (non-profit money only): {_dropped}")

print(investment_by_year_np.head(10))
print()
print(cumulative_by_year_np.head(10))
print()
print(cumulative_rank_np.head(10))


Saved 8d_investment_by_country_and_year_excl_nonprofit -> data_output\analysis\8d_investment_by_country_and_year_excl_nonprofit.csv
Saved 8d_cumulative_investment_by_country_and_year_excl_nonprofit -> data_output\analysis\8d_cumulative_investment_by_country_and_year_excl_nonprofit.csv
Saved 8d_cumulative_investment_rank_excl_nonprofit -> data_output\analysis\8d_cumulative_investment_rank_excl_nonprofit.csv
2020-2025 total: EUR 1,960,488,822 all funders vs EUR 1,711,815,330 excl. non-profit (87.3%).
1 countries drop out entirely (non-profit money only): ['Global']
                              2020          2021          2022          2023  \
Funder Country_split                                                           
Australia             2.060400e+04  4.828994e+06  1.328364e+06  6.063202e+05   
Austria               0.000000e+00  0.000000e+00  6.654340e+05  3.200000e+06   
Belgium               3.600000e+06  0.000000e+00  0.000000e+00  0.000000e+00   
Brazil                0.000000

### 9a. Research category as a % of total funding, by region (+ EU / EU+ / Europe) - overall, and by AP pillar

`'Research category'` is multi-value like `'Funder Country'`/`'Funder region'`, so it gets the
same fractional split - a grant tagged with 3 categories has each get 1/3 of its (region- or
country-weighted) share. Grants with no `'Research category'` at all are excluded entirely, so
`'Total (EUR)'` here is the classified-funding total, not the full region/country total from
1c/3 - percentages are a share of classified funding only.

In [68]:
def category_pct_table(base_split, index_col, weighted_col_name):
    # Grants with no Research category are dropped entirely (not labelled 'Unclassified') -
    # percentages here are a share of classified funding only, so Total (EUR) will be less than
    # the region/country's true total wherever some grants lack a category.
    base = base_split.rename(columns={'weighted_funding': weighted_col_name}).copy()
    exploded = split_multi_value(base, 'Research category', delimiter=',')
    exploded['weighted_funding'] = exploded[weighted_col_name] * exploded['_weight']
    eur = exploded.pivot_table(
        index=index_col, columns='Research category_split', values='weighted_funding', aggfunc='sum', fill_value=0
    )
    eur['Total (EUR)'] = eur.sum(axis=1)
    pct = eur.drop(columns='Total (EUR)').div(eur['Total (EUR)'], axis=0) * 100
    pct['Total (EUR)'] = eur['Total (EUR)']
    return pct.sort_values('Total (EUR)', ascending=False)


def category_pct_by_pillar(base_split, index_col, weighted_col_name):
    pillars = sorted(base_split['AP pillar'].dropna().unique())
    return pd.concat(
        {p: category_pct_table(base_split[base_split['AP pillar'] == p], index_col, weighted_col_name) for p in pillars},
        names=['AP pillar']
    )

In [69]:
region_category_pct = category_pct_table(region_split, 'Funder region_split', 'region_weighted_funding')
save(region_category_pct, '9a_research_category_pct_by_region')
print(region_category_pct)

region_category_pct_by_pillar = category_pct_by_pillar(region_split, 'Funder region_split', 'region_weighted_funding')
save(region_category_pct_by_pillar, '9a_research_category_pct_by_region_and_pillar')
print()
print(region_category_pct_by_pillar)

Saved 9a_research_category_pct_by_region -> data_output\analysis\9a_research_category_pct_by_region.csv
Research category_split  Bioprocess design  Cell culture media  \
Funder region_split                                              
Europe                           10.244087            2.304391   
North America                     5.066339            2.416803   
Asia/Asia Pacific                16.124989            8.850165   
Middle East                       0.000000            0.000000   
Global                           26.362429           12.716364   
South America                     0.359001            4.593517   
Africa                           20.000000            0.000000   

Research category_split  Cell line development  Consumer & market research  \
Funder region_split                                                          
Europe                                1.297469                    6.114026   
North America                         1.647925                    0

In [70]:
tier_category_pct = category_pct_table(tier_split, '_tiers', 'tier_weighted_funding').reindex(['EU', 'EU+', 'Europe'])
save(tier_category_pct, '9a_research_category_pct_by_eu_tier')
print(tier_category_pct)

tier_category_pct_by_pillar = category_pct_by_pillar(tier_split, '_tiers', 'tier_weighted_funding')
save(tier_category_pct_by_pillar, '9a_research_category_pct_by_eu_tier_and_pillar')
print()
print(tier_category_pct_by_pillar)

Saved 9a_research_category_pct_by_eu_tier -> data_output\analysis\9a_research_category_pct_by_eu_tier.csv
Research category_split  Bioprocess design  Cell culture media  \
_tiers                                                           
EU                                9.900253            1.466121   
EU+                              10.245295            2.292878   
Europe                           10.244087            2.304391   

Research category_split  Cell line development  Consumer & market research  \
_tiers                                                                       
EU                                    0.864438                    6.236502   
EU+                                   1.297622                    6.114747   
Europe                                1.297469                    6.114026   

Research category_split  Crop development  End product formulation  \
_tiers                                                               
EU                              

### 9b. Research category as a % of total funding, by country - overall, and by AP pillar

In [71]:
country_category_pct = category_pct_table(country_split, 'Funder Country_split', 'country_weighted_funding')
save(country_category_pct, '9b_research_category_pct_by_country')
print(country_category_pct)

country_category_pct_by_pillar = category_pct_by_pillar(country_split, 'Funder Country_split', 'country_weighted_funding')
save(country_category_pct_by_pillar, '9b_research_category_pct_by_country_and_pillar')
print()
print(country_category_pct_by_pillar)

Saved 9b_research_category_pct_by_country -> data_output\analysis\9b_research_category_pct_by_country.csv
Research category_split  Bioprocess design  Cell culture media  \
Funder Country_split                                             
EU                               11.737117            1.478546   
United States                     5.558337            2.316493   
Denmark                          12.662765            0.522068   
United Kingdom                   12.609021            8.600747   
South Korea                      17.444499            9.342214   
Netherlands                       4.777105            2.669911   
Singapore                        15.089249            3.764120   
Germany                           5.062017            0.710331   
Finland                          14.454670            0.314991   
Canada                            2.640789            2.911334   
Norway                           10.121437            1.736356   
Sweden                            1.

### 9c. Research category as a % of total funding, excl. non-profit funding

Non-profit-excluded twin of 9a/9b, for region, EU / EU+ / Europe, and country. Reuses
`region_split_np` / `tier_split_np` from 4c and `country_split_np` from 3b.

Named **9c**, not 9b - `9b_*` is already the by-country cut of the unfiltered analysis above.

Unlike 4a/4c these tables are **not** reindexed onto 9a/9b's row list. Every figure here is a
percentage of that row's own classified total, and a row left with no funding has no meaningful
percentage (0/0); a row that disappears did so because 100% of its money was non-profit. In the
current data that is `'Global'` in the region cut.


In [72]:
region_category_pct_np = category_pct_table(region_split_np, 'Funder region_split', 'region_weighted_funding')
save(region_category_pct_np, '9c_research_category_pct_by_region_excl_nonprofit')
print(region_category_pct_np)

region_category_pct_by_pillar_np = category_pct_by_pillar(region_split_np, 'Funder region_split', 'region_weighted_funding')
save(region_category_pct_by_pillar_np, '9c_research_category_pct_by_region_and_pillar_excl_nonprofit')
print()
print(region_category_pct_by_pillar_np)


Saved 9c_research_category_pct_by_region_excl_nonprofit -> data_output\analysis\9c_research_category_pct_by_region_excl_nonprofit.csv
Research category_split  Bioprocess design  Cell culture media  \
Funder region_split                                              
Europe                            9.746582            2.495920   
Asia/Asia Pacific                16.126752            8.851132   
North America                     6.234931            2.620023   
Middle East                       0.000000            0.000000   
South America                     0.359001            4.593517   
Africa                           20.000000            0.000000   

Research category_split  Cell line development  Consumer & market research  \
Funder region_split                                                          
Europe                                1.381158                    6.467449   
Asia/Asia Pacific                     4.313112                    2.707260   
North America            

In [73]:
tier_category_pct_np = category_pct_table(tier_split_np, '_tiers', 'tier_weighted_funding').reindex(['EU', 'EU+', 'Europe'])
save(tier_category_pct_np, '9c_research_category_pct_by_eu_tier_excl_nonprofit')
print(tier_category_pct_np)

tier_category_pct_by_pillar_np = category_pct_by_pillar(tier_split_np, '_tiers', 'tier_weighted_funding')
save(tier_category_pct_by_pillar_np, '9c_research_category_pct_by_eu_tier_and_pillar_excl_nonprofit')
print()
print(tier_category_pct_by_pillar_np)


Saved 9c_research_category_pct_by_eu_tier_excl_nonprofit -> data_output\analysis\9c_research_category_pct_by_eu_tier_excl_nonprofit.csv
Research category_split  Bioprocess design  Cell culture media  \
_tiers                                                           
EU                                9.205706            1.589770   
EU+                               9.747853            2.483199   
Europe                            9.746582            2.495920   

Research category_split  Cell line development  Consumer & market research  \
_tiers                                                                       
EU                                    0.908939                    6.740293   
EU+                                   1.381338                    6.468293   
Europe                                1.381158                    6.467449   

Research category_split  Crop development  End product formulation  \
_tiers                                                               
EU

In [74]:
country_category_pct_np = category_pct_table(country_split_np, 'Funder Country_split', 'country_weighted_funding')
save(country_category_pct_np, '9c_research_category_pct_by_country_excl_nonprofit')
print(country_category_pct_np)

country_category_pct_by_pillar_np = category_pct_by_pillar(country_split_np, 'Funder Country_split', 'country_weighted_funding')
save(country_category_pct_by_pillar_np, '9c_research_category_pct_by_country_and_pillar_excl_nonprofit')
print()
print(country_category_pct_by_pillar_np)


Saved 9c_research_category_pct_by_country_excl_nonprofit -> data_output\analysis\9c_research_category_pct_by_country_excl_nonprofit.csv
Research category_split  Bioprocess design  Cell culture media  \
Funder Country_split                                             
EU                               11.737117            1.478546   
United States                     7.302261            2.476801   
United Kingdom                   12.739839            8.689979   
South Korea                      17.444499            9.342214   
Netherlands                       4.777105            2.669911   
Singapore                        15.089249            3.764120   
Germany                           5.062017            0.710331   
Finland                          14.465832            0.315234   
Norway                           10.121437            1.736356   
Canada                            2.795220            3.081586   
Sweden                            1.736568            0.230849   
Denmar

### 9d. Research category by country in EUR (2020-2025)

Same split as 9b, in absolute amounts instead of row percentages. Five CSVs: one overall, plus one
per AP pillar carrying only the categories that occur in that pillar, and every country funded in it.


In [75]:
def category_eur_table(base_split, index_col, weighted_col_name):
    """EUR twin of category_pct_table - identical split logic, absolute amounts instead of row shares."""
    base = base_split.rename(columns={'weighted_funding': weighted_col_name}).copy()
    exploded = split_multi_value(base, 'Research category', delimiter=',')
    exploded['weighted_funding'] = exploded[weighted_col_name] * exploded['_weight']
    eur = exploded.pivot_table(
        index=index_col, columns='Research category_split', values='weighted_funding', aggfunc='sum', fill_value=0
    )
    eur = eur.loc[:, eur.sum() > 0]  # drop categories with no funding in this slice
    eur['Total (EUR)'] = eur.sum(axis=1)
    return eur.sort_values('Total (EUR)', ascending=False)


country_category_eur = category_eur_table(country_split, 'Funder Country_split', 'country_weighted_funding')
save(country_category_eur, '9d_research_category_eur_by_country')
print(country_category_eur.to_string(float_format=lambda v: f'{v:,.0f}'))

# One CSV per AP pillar. Subsetting BEFORE the pivot is what keeps each file to the categories that
# actually occur in that pillar - pivoting first and filtering after would leave all-zero columns.
for pillar in sorted(country_split['AP pillar'].dropna().unique()):
    pillar_eur = category_eur_table(
        country_split[country_split['AP pillar'] == pillar],
        'Funder Country_split', 'country_weighted_funding',
    )
    slug = re.sub(r'[^a-z0-9]+', '-', pillar.lower()).strip('-')
    save(pillar_eur, f'9d_research_category_eur_by_country_{slug}')
    print(f"\n--- {pillar}: {len(pillar_eur)} countries x {len(pillar_eur.columns) - 1} categories, "
          f"EUR {pillar_eur['Total (EUR)'].sum():,.0f} ---")
    print(pillar_eur.to_string(float_format=lambda v: f'{v:,.0f}'))


Saved 9d_research_category_eur_by_country -> data_output\analysis\9d_research_category_eur_by_country.csv
Research category_split  Bioprocess design  Cell culture media  Cell line development  Consumer & market research  Crop development  End product formulation  Feedstocks  Food safety & quality  Health & nutrition  Impact Assessments  Ingredient optimisation       Other  Scaffolding  Strain development  Target molecule selection  Texturization methods  Total (EUR)
Funder Country_split                                                                                                                                                                                                                                                                                                                                                        
EU                              53,339,400           6,719,260                990,773                  39,873,679        12,445,924               61,726,629  26,273

### 9e. Research category by country in EUR, excl. non-profit funding (2020-2025)

9d with non-profit/NGO funders removed - i.e. government money only. Reuses `country_split_np` from
3b and `category_eur_table` from 9d, so both must have been run. Five CSVs, same shapes as 9d.


In [76]:
country_category_eur_np = category_eur_table(country_split_np, 'Funder Country_split', 'country_weighted_funding')
save(country_category_eur_np, '9e_research_category_eur_by_country_excl_nonprofit')
print(country_category_eur_np.to_string(float_format=lambda v: f'{v:,.0f}'))

# Pillar list taken from country_split (not country_split_np) so the same four files are always
# written even if a pillar's funding were entirely non-profit in some future dataset.
for pillar in sorted(country_split['AP pillar'].dropna().unique()):
    pillar_eur_np = category_eur_table(
        country_split_np[country_split_np['AP pillar'] == pillar],
        'Funder Country_split', 'country_weighted_funding',
    )
    slug = re.sub(r'[^a-z0-9]+', '-', pillar.lower()).strip('-')
    save(pillar_eur_np, f'9e_research_category_eur_by_country_{slug}_excl_nonprofit')
    print(f"\n--- {pillar}: {len(pillar_eur_np)} countries x {len(pillar_eur_np.columns) - 1} categories, "
          f"EUR {pillar_eur_np['Total (EUR)'].sum():,.0f} ---")
    print(pillar_eur_np.to_string(float_format=lambda v: f'{v:,.0f}'))

assert country_category_eur_np['Total (EUR)'].sum() <= country_category_eur['Total (EUR)'].sum() + 1


Saved 9e_research_category_eur_by_country_excl_nonprofit -> data_output\analysis\9e_research_category_eur_by_country_excl_nonprofit.csv
Research category_split  Bioprocess design  Cell culture media  Cell line development  Consumer & market research  Crop development  End product formulation  Feedstocks  Food safety & quality  Health & nutrition  Impact Assessments  Ingredient optimisation      Other  Scaffolding  Strain development  Target molecule selection  Texturization methods  Total (EUR)
Funder Country_split                                                                                                                                                                                                                                                                                                                                                       
EU                              53,339,400           6,719,260                990,773                  39,873,679        12,445,924     

### 10. Total funding by sub-production pillar (2020-2025)

Total funding per `'Sub-production pillar'` value, across all AP pillars. Mutually exclusive rows
summing to the grand total, with `(no sub-pillar assigned)` shown explicitly since it is the
majority of funding (Cultivated has no sub-pillar scheme, and most Plant-based grants have none).


In [77]:
SUBPILLAR_COL = 'Sub-production pillar'
total_funding = df[FUNDING_COL].sum()

# Single-valued today, so a plain groupby is correct - no split_multi_value weighting needed.
# Guard in case curation starts packing two values into one cell.
_multi = df[SUBPILLAR_COL].dropna().astype(str).str.contains(r'[;,]')
if _multi.any():
    print(f"WARNING: {_multi.sum()} '{SUBPILLAR_COL}' cells contain ';' or ',' - this analysis "
          f"assumes one value per grant. Switch to split_multi_value before trusting these totals.")

subpillar = df[SUBPILLAR_COL].astype(str).str.strip().where(df[SUBPILLAR_COL].notna())
subpillar_np = df_np[SUBPILLAR_COL].astype(str).str.strip().where(df_np[SUBPILLAR_COL].notna())

by_subpillar = pd.DataFrame({
    'Grants': df.groupby(subpillar)[FUNDING_COL].size(),
    'Total funding (2020-2025, EUR)': df.groupby(subpillar)[FUNDING_COL].sum(),
})
# excl. non-profit column comes from 3b's df_np - run 3b first. Derived from df_np's own column,
# NOT by reindexing a df-based mask: 3b resets df_np's index, so the labels don't line up.
by_subpillar['Total funding excl. non-profit (EUR)'] = df_np.groupby(subpillar_np)[FUNDING_COL].sum()
by_subpillar['% of all funding'] = 100 * by_subpillar['Total funding (2020-2025, EUR)'] / total_funding
by_subpillar = by_subpillar.sort_values('Total funding (2020-2025, EUR)', ascending=False)
by_subpillar.index.name = SUBPILLAR_COL

_blank, _blank_np = df[SUBPILLAR_COL].isna(), df_np[SUBPILLAR_COL].isna()
by_subpillar.loc['(no sub-pillar assigned)'] = [
    int(_blank.sum()), df.loc[_blank, FUNDING_COL].sum(),
    df_np.loc[_blank_np, FUNDING_COL].sum(),
    100 * df.loc[_blank, FUNDING_COL].sum() / total_funding,
]
by_subpillar.loc['ALL GRANTS'] = [len(df), total_funding, df_np[FUNDING_COL].sum(), 100.0]

save(by_subpillar, '10a_funding_by_subpillar')
print(by_subpillar.to_string(float_format=lambda v: f'{v:,.1f}'))


Saved 10a_funding_by_subpillar -> data_output\analysis\10a_funding_by_subpillar.csv
                          Grants  Total funding (2020-2025, EUR)  Total funding excl. non-profit (EUR)  % of all funding
Sub-production pillar                                                                                                   
Broad R&D                  107.0                   299,079,390.6                         270,859,390.6              15.3
Mixed                       72.0                   222,087,099.7                         178,252,191.7              11.3
BF                         232.0                   220,532,980.6                         199,901,311.5              11.2
PF                         124.0                    96,534,976.6                          86,827,970.5               4.9
TF                         105.0                    70,521,864.5                          61,781,461.8               3.6
Socioeconomic               60.0                    16,941,237.4     

### 11. Fermentation-pillar funding by start year x sub-pillar (2020-2025)

Fermentation-pillar grants only, split by `BF` / `PF` / `Mixed`. By START year, same as 1a/2a.


In [78]:
ferment = df[df['AP pillar'] == 'Fermentation']

ferment_year_subpillar = ferment.pivot_table(
    index=SUBPILLAR_COL, columns='Year project started', values=FUNDING_COL, aggfunc='sum', fill_value=0
)
ferment_year_subpillar.columns = ferment_year_subpillar.columns.astype(int)
ferment_year_subpillar = ferment_year_subpillar[sorted(ferment_year_subpillar.columns)]
ferment_year_subpillar.loc['Total'] = ferment_year_subpillar.sum()

save(ferment_year_subpillar, '11_funding_fermentation_by_year_and_subpillar')
print(ferment_year_subpillar.to_string(float_format=lambda v: f'{v:,.0f}'))

# pivot_table drops rows with a blank sub-pillar, so the table total falls short of the pillar total.
_gap = ferment[ferment[SUBPILLAR_COL].isna()]
print(f"\nFermentation pillar total: EUR {ferment[FUNDING_COL].sum():,.0f}. Table covers "
      f"EUR {ferment_year_subpillar.loc['Total'].sum():,.0f}; {len(_gap)} grants "
      f"(EUR {_gap[FUNDING_COL].sum():,.0f}) have no sub-pillar and are excluded.")


Saved 11_funding_fermentation_by_year_and_subpillar -> data_output\analysis\11_funding_fermentation_by_year_and_subpillar.csv
Year project started        2020       2021       2022        2023        2024        2025
Sub-production pillar                                                                     
BF                    10,304,279 25,600,180 38,092,226  56,856,521  18,863,122  70,816,653
Mixed                  9,940,951 14,324,124  2,573,474  41,001,060 115,585,366  38,662,125
PF                     1,506,957  2,744,946  6,454,392  18,331,093  32,075,818  35,421,770
Total                 21,752,187 42,669,250 47,120,092 116,188,673 166,524,305 144,900,549

Fermentation pillar total: EUR 540,403,526. Table covers EUR 539,155,057; 7 grants (EUR 1,248,469) have no sub-pillar and are excluded.


### 12. Funding by start year x AP pillar x country, for a fixed country set (2020-2025)

Same methodology and growth-rate formula as 2c/2d, with a (Country, AP pillar) row index instead of
region. Country set = EU-27 + Norway / UK / Switzerland (EU+), plus US, Canada, Australia,
New Zealand, Brazil, Israel, Japan, South Korea, Singapore, `Global`, and `EU` (supranational
EU-level funding, kept as its own row - it is the largest single funder value in the data).

Every requested country gets a row per pillar even at zero, so the row set is stable across reruns
and pastes into a fixed template. Nine EU-27 members currently have no funding at all
(Bulgaria, Cyprus, Greece, Hungary, Lithuania, Luxembourg, Malta, Romania, Slovenia), as does Italy.


In [79]:
from Funding_analysis_helpers import EU_27

# 'EU' = supranational EU-level funding, kept as its own row (not folded into the member states) -
# same treatment europe_tier_split gives it.
REPORT_COUNTRIES = sorted(EU_27) + [
    'Norway', 'United Kingdom', 'Switzerland',
    'United States', 'Canada', 'Australia', 'New Zealand', 'Brazil',
    'Israel', 'Japan', 'South Korea', 'Singapore',
    'EU', 'Global',
]
YEARS = list(range(2020, 2026))
AP_PILLARS = sorted(df['AP pillar'].dropna().unique())

_unknown = [c for c in REPORT_COUNTRIES if c not in set(country_split['Funder Country_split'])]
if _unknown:
    print(f"No funding found for: {_unknown} - kept as zero rows. Check spelling against "
          f"'Funder Country' if any of these were expected to have grants.")

country_pillar_year = country_split[country_split['Funder Country_split'].isin(REPORT_COUNTRIES)].pivot_table(
    index=['Funder Country_split', 'AP pillar'], columns='Year project started',
    values='weighted_funding', aggfunc='sum', fill_value=0,
)
country_pillar_year.columns = country_pillar_year.columns.astype(int)

# Reindex onto the full (country x pillar) grid so absent combinations read as 0 rather than vanishing.
_grid = pd.MultiIndex.from_product([REPORT_COUNTRIES, AP_PILLARS], names=['Funder Country', 'AP pillar'])
country_pillar_year = country_pillar_year.reindex(index=_grid, columns=YEARS, fill_value=0).fillna(0)
country_pillar_year['Total'] = country_pillar_year[YEARS].sum(axis=1)

save(country_pillar_year, '12a_funding_by_year_country_and_pillar')
print(country_pillar_year.to_string(float_format=lambda v: f'{v:,.0f}'))

# Growth rates, same formula as 2b/2d. yoy_growth needs the year columns only, not 'Total'.
growth_country_pillar = country_pillar_year[YEARS].apply(yoy_growth, axis=1)
growth_country_pillar['Average growth rate (2021-2025)'] = growth_country_pillar.iloc[:, 1:].mean(axis=1)
save(growth_country_pillar, '12b_growth_rates_by_country_and_pillar')
print()
print(growth_country_pillar.to_string(float_format=lambda v: f'{v:,.2f}'))

print(f"\nTable covers EUR {country_pillar_year['Total'].sum():,.0f} of "
      f"EUR {country_split['weighted_funding'].sum():,.0f} total "
      f"({100 * country_pillar_year['Total'].sum() / country_split['weighted_funding'].sum():.1f}%) - "
      f"the remainder sits in countries outside REPORT_COUNTRIES.")


No funding found for: ['Bulgaria', 'Cyprus', 'Greece', 'Hungary', 'Lithuania', 'Luxembourg', 'Malta', 'Romania', 'Slovenia'] - kept as zero rows. Check spelling against 'Funder Country' if any of these were expected to have grants.
Saved 12a_funding_by_year_country_and_pillar -> data_output\analysis\12a_funding_by_year_country_and_pillar.csv
Year project started               2020       2021       2022       2023       2024       2025       Total
Funder Country AP pillar                                                                                  
Austria        Cross-cutting          0          0          0          0          0    897,906     897,906
               Cultivated             0          0          0          0          0          0           0
               Fermentation           0          0          0  3,200,000          0          0   3,200,000
               Plant-based            0          0    665,434          0          0    447,261   1,112,695
Belgium       

### 13. Research Question No. 18

Evaluate how public R&I grants act as a catalyst for private investment by correlating specific public grants to private investment data. This analysis aims to build on the previous section by describing the link between public support and downstream private capital investment and IP generation activity. Depending on data availability, it will also analyse the degree to which not receiving public funding support has been a hindrance to companies in relation to intellectual property generation and commercial activity.

In [80]:
# Read in private investment data that was collected by other GFI colleagues
# private investment was collected by year of deal
private2025 = pd.read_csv('data_output/private_investment_2025.csv')
private2024 = pd.read_csv('data_output/private_investment_2024.csv')
private2023 = pd.read_csv('data_output/private_investment_2023.csv')
private2022 = pd.read_csv('data_output/private_investment_2022.csv')
private2021 = pd.read_csv('data_output/private_investment_2021.csv')
private2020 = pd.read_csv('data_output/private_investment_2020.csv')

In [81]:
# combine all private investment tables in one
private = pd.concat([private2025, private2024, private2023, private2022, private2021, private2020])
private = private[private["Net0i Organisation ID"].notna()]
private["Net0i Organisation ID"] = private["Net0i Organisation ID"].astype(int)

In [82]:
# Check whether the names and organisation ID's match (= if one ID corresponds to one name only)
private.drop_duplicates(['Organisation name', 'Net0i Organisation ID'])['Net0i Organisation ID'].value_counts()

# after cleaning up some names, all ID's are unique except for two ID's that correspond to companies that changed their name
# Nutrumami -> SUMM Ingredients (ID: 153164); Re:Meat -> Curve (ID: 124873)

Net0i Organisation ID
153164    2
124873    2
31814     1
486       1
55931     1
         ..
179082    1
496       1
128227    1
43723     1
73048     1
Name: count, Length: 279, dtype: int64

In [83]:
df_np

,Title,Abstract,Original title,Database,Total amount,Gov & NP contribution,Currency,Total amount (USD),Gov & NP contribution (USD),Total amount (EUR),...,Unnamed: 95,Unnamed: 96,Unnamed: 97,Unnamed: 98,Unnamed: 99,Unnamed: 100,Unnamed: 101,Unnamed: 102,Unnamed: 103,Unnamed: 104
0,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,7524240,7524240.00,5.585400e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, a...",NAPIC: National Alternative Protein Innovation...,airtable,38000000,16001352,GBP,48859450,18697500.00,4.522000e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Green technology for plant-based food (GreenPl...,To maintain national food self-sufficiency and...,GreenPlantFood - Green technology for plant-ba...,airtable,27600000,27600000,NOK,2604048,2604048.00,2.401200e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SYLPLANT,"Arbiom, an agricultural-biotech company develo...",NaN,airtable,23000000,14000000,EUR,25829000,15722000.00,2.300000e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,University of Bath Cellular Agriculture Manufa...,Imagine being able to manufacture food anywher...,Cellular Agriculture Manufacturing Hub,airtable,21214469,12302109,GBP,26130285,15655848.00,2.524522e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2238,Establishment of a process-specific risk facto...,NaN,NaN,GFI public investment database,40000000,40000000,KRW,28452,28452.00,2.600000e+04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2239,Study on the Safety Assessment Methods for Nov...,"[machine translated] ""Applying precision ferme...",NaN,GFI public investment database,60000001,60000001,KRW,41400,41400.00,3.900000e+04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2240,2025 High Value-Added Food Technology Developm...,"1,500,000,000 KRW each on the research topics ...",NaN,GFI public investment database,3000000000,3000000000,KRW,2063400,2063400.00,1.950000e+06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2241,Research on improving safety assessment of foo...,"식품등안전관리 - 4) 선제적 위해요인 안전관리 (""Food and Product ...",NaN,GFI public investment database,200000000,200000000,KRW,154380,154380.00,1.400000e+05,...,3.0,NaN,False,False,False,False,ok,tool_use,260930.0,manual_review


#### 13a. Normalise organisation names and generate match candidates

Matches **from** the curated private-investment company list **into** the funding data's
organisation text, rather than trying to resolve every funding org name into a canonical entity -
201 targeted lookups instead of ~1,200 messy names, and every candidate stays reviewable.

Matching and fractional weighting are deliberately decoupled: matching searches the *unsplit* text
(so a name shredded by a comma inside it is still found), while the recipient split is used only
for the 1/n denominator.

Tiers: `exact` (normalised equal, or equal ignoring spaces) and `contains` on a multi-token name
are auto-accepted; `contains` on a **single-token** name and anything `fuzzy` go to a review file.
Single-token names are the false-positive risk - 95 of the 201 companies have one-word names, and
e.g. `Multus` legitimately matches `Multus Biotechnology Limited` but not `Multus Media`.

The review file is written **without a run timestamp** so hand edits survive re-runs. Edit the
`accept` column (True/False) and re-run 13b to pick the decisions up.


In [84]:
import unicodedata
from rapidfuzz import fuzz

REVIEW_FILE = Path('data_output/company_match_review.csv')  # deliberately NOT timestamped
FUZZ_MIN = 88

# LEGAL FORMS ONLY. Descriptive words ('technologies', 'labs', 'group') are deliberately NOT
# stripped: removing them reduced 'Roslin Technologies' to the generic token 'roslin', which then
# matched 'Instytut Genetyki Roslin Polskiej Akademii Nauk'.
LEGAL_FORMS = (r'(ltd|limited|inc|incorporated|llc|plc|gmbh|mbh|ug|ag|bv|nv|ab|oyj|oy|as|asa|aps|'
               r'sa|sl|slu|srl|sas|sarl|spa|kg|kft|doo|ou|ehf|sia|pte|pty|co|corp|corporation|company)')


def normalise_org(name):
    """Lowercase, de-accent, drop parentheticals/punctuation, strip legal forms. '' if unusable."""
    if pd.isna(name):
        return ''
    s = str(name).replace('&amp;', '&').replace('&', ' and ')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()  # 'Sophie’s' -> "sophie's"
    s = s.lower()
    s = re.sub(r'\(.*?\)', ' ', s)
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    for _ in range(2):  # handles stacked forms like 'foo bv ltd'
        s = re.sub(rf'\b{LEGAL_FORMS}\b', ' ', s)
    s = re.sub(r'\bthe\b', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def _despace(s):
    return s.replace(' ', '')  # 'onego bio' == 'onegobio'


def split_collaborators(value):
    """Best-effort split of 'Collaborator institutions'. Delimiters are inconsistent (';' 109 rows,
    ',' 159, ' and ' 72, '&' 26) and commas also occur INSIDE names, so prefer ';' when present and
    only fall back to ','. Used for the 1/n denominator only - never for matching."""
    if pd.isna(value):
        return []
    t = str(value).replace('&amp;', '&')
    if ';' in t or '\n' in t:
        parts = re.split(r';|\n', t)
    elif ',' in t:
        parts = re.split(r',', t)
    else:
        parts = re.split(r'\s+and\s+|\s*/\s*', t)
    return [x.strip() for x in parts if x.strip()]


# --- recipients per grant, from df_np (non-profit funders already excluded) ---
# 'Multiple organisation recipients' is NOT used as a gate: it is 2,166 NaN / 133 No / 104 Yes plus
# ~19 rows holding country lists, and 396 grants list collaborators while it is not 'Yes'.
_recip_rows = []
for _idx, _r in df_np.iterrows():
    _pi = _r['PI organisation']
    _raw_collab = _r['Collaborator institutions']
    _recips = ([_pi] if pd.notna(_pi) else []) + split_collaborators(_raw_collab)
    # norm_blob is built from the UNSPLIT text on purpose - it is the fallback that catches a name
    # the split damaged, so it must not itself go through the split. Rejoining the split parts would
    # nearly work (delimiters collapse to spaces either way), but the ' and ' / '/' fallback branch
    # CONSUMES its delimiter: 'Flour Milling and Baking Research Association' loses its 'and', so a
    # company whose real name contains 'and' could not be recovered here. 6 grants differ.
    _blob_parts = ([str(_pi)] if pd.notna(_pi) else []) + ([str(_raw_collab)] if pd.notna(_raw_collab) else [])
    _recip_rows.append({
        'grant_idx': _idx,
        'recipients': _recips,
        'norm_recipients': [normalise_org(x) for x in _recips],
        'norm_blob': normalise_org(' ; '.join(_blob_parts)),
        'n_recipients': max(len(_recips), 1),
    })
grant_recipients = pd.DataFrame(_recip_rows)
print(f"{len(grant_recipients)} grants; {(grant_recipients['recipients'].map(len) > 0).sum()} name at "
      f"least one recipient. Recipients per grant: median "
      f"{int(grant_recipients['n_recipients'].median())}, max {grant_recipients['n_recipients'].max()}.")

# --- fuzzy engine: rapidfuzz if available, stdlib difflib otherwise (different scores, same tiers) ---

def _ratio(a, b): return fuzz.token_set_ratio(a, b)
FUZZ_ENGINE = 'rapidfuzz.token_set_ratio'

print(f'Fuzzy engine: {FUZZ_ENGINE}')

# --- company search list: EVERY name an ID has ever carried, not one row per ID ---
# Companies get renamed (e.g. 124873 Re:Meat -> Curve, 153164 Nutrumami -> SUMM Ingredients) and in
# both of those cases only the OLD name appears in the funding data - the grant predates the rename.
# Searching just the current name loses the match, so every (id, name) pair is searched and results
# attributed back to the id. 13b de-duplicates on (org_id, grant_idx), so an alias cannot double-count.
companies = (private[['Net0i Organisation ID', 'Organisation name']]
             .drop_duplicates()
             .rename(columns={'Net0i Organisation ID': 'org_id', 'Organisation name': 'org_name'}))
companies['org_norm'] = companies['org_name'].map(normalise_org)
companies['n_tokens'] = companies['org_norm'].str.split().str.len()
companies = (companies[companies['org_norm'] != '']
             .drop_duplicates(['org_id', 'org_norm']).reset_index(drop=True))
_n_ids = companies['org_id'].nunique()
_aliases = len(companies) - _n_ids
print(f"{_n_ids} private-investment companies searched under {len(companies)} name variants "
      f"({_aliases} renamed/alias); {(companies['n_tokens'] == 1).sum()} variants are single-token "
      f"(never auto-accepted).")

_cands = []
for _c in companies.itertuples(index=False):
    _cn, _cd = _c.org_norm, _despace(_c.org_norm)
    _pat = re.compile(rf'(?<![a-z0-9]){re.escape(_cn)}(?![a-z0-9])')
    for _g in grant_recipients.itertuples(index=False):
        _best = None
        for _rn, _raw in zip(_g.norm_recipients, _g.recipients):
            if not _rn:
                continue
            if _rn == _cn or _despace(_rn) == _cd:
                _best = ('exact', 100.0, _raw)
                break
            if _best is None and _pat.search(_rn):
                _best = ('contains', 100.0, _raw)
        if _best is None and _pat.search(_g.norm_blob):
            # name survived only in the unsplit text - a comma inside it shredded the split
            _best = ('contains_blob', 100.0, ' ; '.join(str(x) for x in _g.recipients)[:140])
        if _best is None:
            for _rn, _raw in zip(_g.norm_recipients, _g.recipients):
                if not _rn:
                    continue
                _sc = _ratio(_cn, _rn)
                if _sc >= FUZZ_MIN and (_best is None or _sc > _best[1]):
                    _best = ('fuzzy', _sc, _raw)
        if _best:
            _tier, _score, _matched = _best
            _cands.append({
                'org_id': _c.org_id, 'org_name': _c.org_name, 'org_norm': _cn,
                'n_tokens': _c.n_tokens, 'grant_idx': _g.grant_idx, 'tier': _tier,
                'score': round(_score, 1), 'matched_text': _matched,
                'n_recipients': _g.n_recipients,
                'accept': _tier == 'exact' or (_tier in ('contains', 'contains_blob') and _c.n_tokens >= 2),
            })

match_candidates = pd.DataFrame(_cands)
match_candidates['needs_review'] = ~match_candidates['accept']
print()
print(match_candidates.groupby(['tier', 'accept']).size().to_string())
print(f"\n{match_candidates['org_id'].nunique()} of {_n_ids} companies have at least one "
      f"candidate; {match_candidates.loc[match_candidates['accept'], 'org_id'].nunique()} are "
      f"auto-accepted. {int(match_candidates['needs_review'].sum())} pairs need review.")

if REVIEW_FILE.exists():
    print(f"\n{REVIEW_FILE} already exists - NOT overwritten. Delete it to regenerate from scratch.")
else:
    match_candidates.to_csv(REVIEW_FILE, index=False)
    print(f"\nWrote {REVIEW_FILE}. Edit the 'accept' column for the rows where needs_review is True, "
          f"then re-run 13b.")

print()
print(match_candidates[match_candidates['needs_review']]
      [['org_name', 'n_tokens', 'tier', 'score', 'matched_text']].to_string())


2243 grants; 2109 name at least one recipient. Recipients per grant: median 1, max 159.
Fuzzy engine: rapidfuzz.token_set_ratio
279 private-investment companies searched under 281 name variants (2 renamed/alias); 129 variants are single-token (never auto-accepted).

tier      accept
contains  False      29
          True       16
exact     True      165
fuzzy     False      27

106 of 279 companies have at least one candidate; 84 are auto-accepted. 56 pairs need review.

data_output\company_match_review.csv already exists - NOT overwritten. Delete it to regenerate from scratch.

                       org_name  n_tokens      tier  score                                                                                                                                                                                            matched_text
26          Roslin Technologies         2     fuzzy  100.0                                                                                                 

#### 13b. Public funding per matched company, by year

Applies the accepted matches with fractional counting: a grant's amount is split evenly across its
recipients (`1 / n_recipients`), and the matched company takes one share. If a company matched a
grant more than once (e.g. named as both PI and collaborator) it is de-duplicated to one share.

Reads `company_match_review.csv` if it exists so hand decisions win; otherwise falls back to the
auto-accepted tiers.


In [85]:
if REVIEW_FILE.exists():
    _decisions = pd.read_csv(REVIEW_FILE)
    _decisions['accept'] = _decisions['accept'].astype(str).str.strip().str.lower().isin(['true', '1', 'yes'])
    print(f'Using hand-reviewed decisions from {REVIEW_FILE}.')
else:
    _decisions = match_candidates.copy()
    print('No review file found - using auto-accepted tiers only.')

# Canonical name per Net0i ID, taken from the most recent deal. Two IDs carry two names because the
# company was renamed (Re:Meat -> Curve, Nutrumami -> SUMM Ingredients), so org_id is the ONLY safe
# join key - joining on the name as well would split those companies into separate rows.
ORG_NAME = (private.sort_values('Deal year')
            .drop_duplicates('Net0i Organisation ID', keep='last')
            .set_index('Net0i Organisation ID')['Organisation name'])


def _org_attribute(column):
    """Per-COMPANY lookup of a company-level attribute, from the most recent deal that records one.
    HQ country/continent belong to the company, not to a (company, year) pair, so a year in which a
    company has public funding but no private deal must still carry them - carrying them through the
    groupby alone leaves those rows NA (e.g. Bruno Cell: Italy on its 2020 deal, blank on its 2024
    grant-only row)."""
    have = private[private[column].notna()]
    return (have.sort_values('Deal year')
            .drop_duplicates('Net0i Organisation ID', keep='last')
            .set_index('Net0i Organisation ID')[column])


ORG_COUNTRY = _org_attribute('Organisation HQ country')
ORG_CONTINENT = _org_attribute('Organisation continent')

accepted = _decisions[_decisions['accept']].copy()
# One share per (company, grant): a company named as both PI and collaborator must not double-count.
accepted = accepted.drop_duplicates(['org_id', 'grant_idx'])
print(f"{len(accepted)} accepted (company, grant) pairs covering "
      f"{accepted['org_id'].nunique()} companies and {accepted['grant_idx'].nunique()} grants.")

_grant_cols = df_np[[FUNDING_COL, 'Year project started']].copy()
public_matched = accepted.merge(_grant_cols, left_on='grant_idx', right_index=True, how='left')
public_matched['company_share_eur'] = public_matched[FUNDING_COL] / public_matched['n_recipients']
public_matched['Year'] = public_matched['Year project started'].astype(int)

public_by_company_year = (public_matched.groupby(['org_id', 'Year'])
                          .agg(public_funding_eur=('company_share_eur', 'sum'),
                               grants=('grant_idx', 'nunique'))
                          .reset_index())
print(f"\nPublic funding attributed to matched companies: "
      f"EUR {public_by_company_year['public_funding_eur'].sum():,.0f} "
      f"({100 * public_by_company_year['public_funding_eur'].sum() / df_np[FUNDING_COL].sum():.1f}% of df_np).")
print(public_by_company_year.assign(org_name=lambda d: d['org_id'].map(ORG_NAME))
      .sort_values('public_funding_eur', ascending=False).head(15).to_string(index=False))


Using hand-reviewed decisions from data_output\company_match_review.csv.
222 accepted (company, grant) pairs covering 98 companies and 181 grants.

Public funding attributed to matched companies: EUR 110,958,752 (6.5% of df_np).
 org_id  Year  public_funding_eur  grants                     org_name
 165558  2024        9.605077e+06       2                      Labfarm
  56605  2022        8.507457e+06       1                  Melt&Marble
  31687  2020        8.352000e+06       1          The Protein Brewery
 104301  2022        5.999858e+06       1                   Herlab Ltd
  29755  2022        5.440049e+06       5                       Multus
  88765  2020        5.140274e+06       1                Extracellular
  31807  2024        4.499428e+06       1      Time-Travelling Milkman
 101471  2025        4.373420e+06       2             New Wave Biotech
   8777  2025        2.784086e+06       3                BioTech Foods
  40342  2023        2.695038e+06       4          Roslin Tec

In [86]:
public_by_company_year

,org_id,Year,public_funding_eur,grants
0,486,2023,3.004208e+05,1
1,486,2024,1.060667e+06,2
2,489,2021,4.351890e+05,2
3,489,2023,2.561608e+06,1
4,489,2024,7.972200e+04,1
...,...,...,...,...
161,124873,2024,3.000000e+05,1
162,127895,2023,9.984700e+04,1
163,153164,2021,3.116250e+05,1
164,153339,2024,1.992960e+05,1


#### 13c. Public vs private investment by company and year

Private deal amounts are USD; converted to EUR with a year-end rate per deal year (same
Frankfurter source as 3a). Deals with no recorded amount are counted in `deals` but contribute 0 to
`private_investment_eur` - check the coverage line before using the ratio.

Rows are the union of (company, year) seen in either dataset, so a company with public funding but
no private deal that year appears with a zero, and vice versa.


In [87]:
# --- private side: USD -> EUR per deal year ---
private_deals = private.copy()
def _to_number(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False), errors='coerce')

private_deals['deal_amount_usd'] = _to_number(private_deals['Deal amount USD'])
private_deals['deal_amount_orig'] = _to_number(private_deals['Deal amount original currency'])
private_deals['Year'] = private_deals['Deal year'].astype(int)

# A deal already denominated in EUR is taken at its ORIGINAL amount - no conversion. The vendor's
# USD figure was converted at the deal-date rate (implied USD/EUR ranges 1.09-1.22 within 2020
# alone), so round-tripping it back through one year-end rate would inject error into a number that
# is already exact. Everything else converts from USD at a year-end rate, as elsewhere in this
# notebook.
_orig_ccy = private_deals['Deal original currency'].astype(str).str.strip().str.upper()
_is_eur = _orig_ccy.eq('EUR') & private_deals['deal_amount_orig'].notna()

_rates = {int(y): fetch_usd_to_eur_rate(f'{int(y)}-12-29') for y in sorted(private_deals['Year'].unique())}
print('USD->EUR year-end rates (used only for non-EUR deals):', _rates)

private_deals['deal_amount_eur'] = np.where(
    _is_eur,
    private_deals['deal_amount_orig'],
    private_deals['deal_amount_usd'] * private_deals['Year'].map(_rates),
)
private_deals['amount_source'] = np.where(
    _is_eur, 'original EUR',
    np.where(private_deals['deal_amount_usd'].notna(), 'converted from USD', 'no amount'))

print(private_deals['amount_source'].value_counts().to_string())
_missing = (private_deals['amount_source'] == 'no amount').sum()
print(f"{_missing} of {len(private_deals)} deals ({100 * _missing / len(private_deals):.0f}%) have no "
      f"recorded amount and contribute 0 EUR - treat the ratio as a lower bound on private capital.")

private_by_company_year = (private_deals.groupby(['Net0i Organisation ID', 'Year', 'Organisation HQ country', 'Organisation continent'])
                           .agg(private_investment_eur=('deal_amount_eur', 'sum'),
                                deals=('Net0i Organisation ID', 'size'),
                                deals_with_amount=('deal_amount_eur', 'count'))
                           .reset_index()
                           .rename(columns={'Net0i Organisation ID': 'org_id', 
                                            'Organisation HQ country': 'country',
                                            'Organisation continent': 'continent'}))

# Outer join on org_id + Year ONLY (see ORG_NAME note in 13b), so public-only and private-only
# (company, year) rows both survive; the name is attached afterwards.
public_vs_private = (public_by_company_year
                     .merge(private_by_company_year, on=['org_id', 'Year'], how='outer')
                     .fillna({'public_funding_eur': 0, 'grants': 0,
                              'private_investment_eur': 0, 'deals': 0, 'deals_with_amount': 0}))
public_vs_private.insert(1, 'org_name', public_vs_private['org_id'].map(ORG_NAME))
# Backfill company-level attributes on rows that came only from the public side (no deal that year).
for _col, _lookup in (('country', ORG_COUNTRY), ('continent', ORG_CONTINENT)):
    _before = public_vs_private[_col].isna().sum()
    public_vs_private[_col] = public_vs_private[_col].fillna(public_vs_private['org_id'].map(_lookup))
    _after = public_vs_private[_col].isna().sum()
    print(f"{_col}: filled {_before - _after} of {_before} blank rows from the company's other rows"
          f"{f'; {_after} still blank' if _after else ''}.")
public_vs_private[['grants', 'deals', 'deals_with_amount']] = \
    public_vs_private[['grants', 'deals', 'deals_with_amount']].astype(int)
public_vs_private = public_vs_private.sort_values(['org_name', 'Year']).reset_index(drop=True)

assert public_vs_private['org_name'].notna().all(), 'org_id present in funding matches but not in private data'
assert not public_vs_private.duplicated(['org_id', 'Year']).any(), 'duplicate (company, year) rows'

save(public_vs_private, 'Q18_public_vs_private_by_company_and_year', index=False)
print()
print(public_vs_private.to_string(index=False, float_format=lambda v: f'{v:,.0f}'))

_yr = public_vs_private.groupby('Year')[['public_funding_eur', 'private_investment_eur']].sum()
_yr['private_per_public_eur'] = _yr['private_investment_eur'] / _yr['public_funding_eur']
print()
_pub_yrs = sorted(public_by_company_year['Year'].unique())
_prv_yrs = sorted(private_by_company_year['Year'].unique())
print(f'Year totals across matched companies. NOT a causal ratio - public and private are')
print(f'contemporaneous here, with no lag applied. Public years {_pub_yrs[0]}-{_pub_yrs[-1]}, '
      f'private years {_prv_yrs[0]}-{_prv_yrs[-1]};')
print(f'any year present in only one of the two gives a meaningless ratio.')
print(_yr.to_string(float_format=lambda v: f'{v:,.2f}'))


USD->EUR year-end rates (used only for non-EUR deals): {2020: 0.81573, 2021: 0.88472, 2022: 0.93906, 2023: 0.90498, 2024: 0.95831, 2025: 0.84991}
amount_source
no amount             468
original EUR          347
converted from USD    275
468 of 1090 deals (43%) have no recorded amount and contribute 0 EUR - treat the ratio as a lower bound on private capital.
country: filled 68 of 68 blank rows from the company's other rows.
continent: filled 68 of 68 blank rows from the company's other rows.
Saved Q18_public_vs_private_by_company_and_year -> data_output\analysis\Q18_public_vs_private_by_company_and_year.csv

 org_id                       org_name  Year  public_funding_eur  grants        country continent  private_investment_eur  deals  deals_with_amount
  68536                       21St.Bio  2021                   0       0        Denmark    Europe              86,000,000      1                  1
 179045                          5Yes!  2021                   0       0         Russia

In [88]:
public_vs_private

,org_id,org_name,Year,public_funding_eur,grants,country,continent,private_investment_eur,deals,deals_with_amount
0,68536,21St.Bio,2021,0.0,0,Denmark,Europe,8.600000e+07,1,1
1,179045,5Yes!,2021,0.0,0,Russia,Europe,0.000000e+00,1,0
2,179046,Aak,2021,0.0,0,Sweden,Europe,4.000000e+08,1,1
3,179046,Aak,2022,0.0,0,Sweden,Europe,2.539907e+08,3,3
4,42893,Aberyne,2022,0.0,0,France,Europe,0.000000e+00,1,0
...,...,...,...,...,...,...,...,...,...,...
687,102467,YOFI,2021,0.0,0,France,Europe,1.105900e+05,1,1
688,128294,Yeastup,2022,0.0,0,Switzerland,Europe,8.188603e+05,1,1
689,128294,Yeastup,2024,0.0,0,Switzerland,Europe,1.916627e+07,2,2
690,12254,Yumgo,2022,0.0,0,France,Europe,1.500000e+06,1,1


### 14. HORIZON money

In [89]:
horizon = pd.read_csv("data_output/HORIZON_funding.csv")

In [90]:
horizon

,Title,Abstract,Total amount,Gov & NP contribution,Currency,Funding decision,URL for announcement,Identification code,Project start date,Funder name,...,microalgae,Award purpose,End product type,sub-end product,Research category,Year project started,End date,recipient_institutions,recipient_countries,recipient_funding_amount
0,Like-A-Pro: From niche to mainstream - alterna...,The interest of Europeans in alternative prote...,13909093.00,11999660.00,EUR,Awarded,https://cordis.europa.eu/project/id/101083961,grant.13029417,2022-11-01 0:00:00,European Commission,...,NaN,Research and development; Education and traini...,Agnostic,NaN,"End product formulation, Consumer & market res...",2022.0,2026.0,CENTRO NACIONAL DE TECNOLOGIA Y SEGURIDAD ALIM...,Spain; Norway; Germany; Denmark; Belgium; Belg...,1 341 092.05; 978 201.25; 809 937.50; 551 775....
1,HealthFerm: Innovative pulse and cereal-based ...,Food fermentation is a natural process in whic...,11303944.75,11303944.75,EUR,Awarded,https://cordis.europa.eu/project/id/101060247/...,grant.12941144,2022-09-01 0:00:00,European Commission,...,NaN,Research and development,Agnostic,NaN,"Strain development, Ingredient optimisation, E...",2022.0,2026.0,KATHOLIEKE UNIVERSITEIT LEUVEN; VIB VZW; CHALM...,Belgium; Belgium; Sweden; Finland; Belgium; Fi...,2 046 148.42; 987 750.00; 826 475.00; 754 206....
2,Harnessing the microbial potential of fermente...,Food and beverage fermentation dates back 7 00...,10815148.00,10815148.00,EUR,Awarded,https://cordis.europa.eu/project/id/101060218,NaN,2023-03-01 0:00:00,European Commission,...,NaN,Research and development,Dairy,Yoghurt and fermented dairy,"End product formulation, Health & nutrition",2023.0,2028.0,INSTITUT NATIONAL DE RECHERCHE POUR L'AGRICULT...,France; Ireland; Italy; Spain; Italy; Italy; B...,2 848 336.25; 1 277 277.50; 1 072 187.50; 919 ...
3,"Gap resolutIon in sAfety, NuTritional, alLerge...",NaN,10347863.00,10272862.00,EUR,Awarded,https://cordis.europa.eu/project/id/101059632,grant.12960620,2022-09-01 0:00:00,European Commission,...,NaN,Research and development,Agnostic,NaN,"Food safety & quality, Health & nutrition",2022.0,2026.0,STICHTING WAGENINGEN RESEARCH; TEKNOLOGIAN TUT...,Netherlands; Finland; Germany; Netherlands; Fr...,1 724 397.50; 759 389.00; 736 000.00; 719 825....
4,new VALue landscapes for plant PROtein Pathway...,The current European plant-protein landscape i...,9580339.88,8658160.51,EUR,Awarded,https://cordis.europa.eu/project/id/101059824,grant.12960983,2022-09-01 0:00:00,European Commission,...,NaN,Research and development; Networking,Agnostic,NaN,Impact Assessments,2022.0,2026.0,TEAGASC - AGRICULTURE AND FOOD DEVELOPMENT AUT...,Ireland; Denmark; Italy; Serbia; Netherlands; ...,1 623 293.63; 1 201 237.50; 697 750.00; 606 95...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162,FunSea,One of our most pressing societal challenges t...,NaN,NaN,NaN,Awarded,https://bluepartnership.eu/projects/functional...,need to check,2024-04-01 0:00:00,European Commission,...,NaN,Research and development,Agnostic,NaN,"Ingredient optimisation, End product formulati...",2024.0,2027.0,NaN,NaN,NaN
163,AlgFlavor: Enhancing algal biomass consumer ac...,As society shifts away from animal-based diets...,1666667.00,1666667.00,EUR,Awarded,https://bluepartnership.eu/news/24-projects-re...,NaN,2025-05-01 0:00:00,European Commission,...,NaN,Research and development,Fish and seafood,NaN,"Bioprocess design, Ingredient optimisation, En...",2025.0,2028.0,NaN,NaN,NaN
164,SEANERGIES: Cell-cultured octopus production t...,The world’s population is projected to surpass...,1666667.00,1666667.00,EUR,Awarded,https://bluepartnership.eu/news/24-projects-re...,NaN,2025-05-01 0:00:00,European Commission,...,NaN,Research and development,Fish and seafood,NaN,"Cell line development, Cell culture media, Bio...",2025.0,2028.0,NaN,NaN,NaN
165,BIOVAL: Microalgal biomass valorization for su...,The main aim of this proposal is the de

#### 14a. HORIZON funding by sub-funder and recipient country

`recipient_institutions`, `recipient_countries` and `recipient_funding_amount` are three parallel
`;`-separated lists - position *i* in each refers to the same recipient - so they are zipped
positionally rather than split independently. No fractional counting: each recipient's exact
amount is used.

Amounts use a space as the thousands separator (`1 341 092.05`), so the spaces are stripped before
parsing. Rows whose three lists disagree in length are reported and skipped rather than silently
mis-zipped.


In [91]:
def _split_semicolon(value):
    """'A; B; C' -> ['A', 'B', 'C']. Empty list for a blank cell."""
    if pd.isna(value):
        return []
    return [x.strip() for x in str(value).split(';') if x.strip() != '']


def _parse_amount(value):
    """'1 341 092.05' -> 1341092.05. Space (incl. non-breaking) is the thousands separator here,
    '.' the decimal point."""
    return pd.to_numeric(re.sub(r'[\s\u00a0\u202f]', '', str(value)), errors='coerce')


RECIPIENT_COLS = ['recipient_institutions', 'recipient_countries', 'recipient_funding_amount']

_recipient_rows = []
_no_breakdown = 0
_misaligned = []
for _idx, _r in horizon.iterrows():
    _inst = _split_semicolon(_r['recipient_institutions'])
    _ctry = _split_semicolon(_r['recipient_countries'])
    _amts = _split_semicolon(_r['recipient_funding_amount'])
    if not _ctry:
        _no_breakdown += 1
        continue
    # The three lists are positionally aligned, so a length mismatch would silently pair the wrong
    # country with the wrong amount - skip and report instead of zipping to the shortest list.
    if not (len(_inst) == len(_ctry) == len(_amts)):
        _misaligned.append((_idx, len(_inst), len(_ctry), len(_amts)))
        continue
    for _i, (_institution, _country, _amount) in enumerate(zip(_inst, _ctry, _amts)):
        _recipient_rows.append({
            'grant_idx': _idx,
            'Title': _r['Title'],
            'Sub-funder': _r['Sub-funder'],
            'recipient_institution': _institution,
            'recipient_country': _country,
            'recipient_funding_eur': _parse_amount(_amount),
        })

horizon_recipients = pd.DataFrame(_recipient_rows)
horizon_recipients['Sub-funder'] = horizon_recipients['Sub-funder'].fillna('NA')

print(f"{len(horizon)} HORIZON grants -> {len(horizon_recipients)} recipient entries "
      f"across {horizon_recipients['recipient_country'].nunique()} countries and "
      f"{horizon_recipients['Sub-funder'].nunique()} sub-funders.")
print(f"{_no_breakdown} grants have no recipient breakdown and are excluded.")
if _misaligned:
    print(f"WARNING: {len(_misaligned)} grants skipped - the three lists disagree in length "
          f"(grant_idx, n_inst, n_country, n_amount): {_misaligned}")
_unparsed = int(horizon_recipients['recipient_funding_eur'].isna().sum())
if _unparsed:
    print(f'WARNING: {_unparsed} recipient amounts could not be parsed and count as 0.')

# One number per sub-funder per country.
horizon_subfunder_country = horizon_recipients.pivot_table(
    index='Sub-funder', columns='recipient_country', values='recipient_funding_eur',
    aggfunc='sum', fill_value=0,
)
horizon_subfunder_country['Total'] = horizon_subfunder_country.sum(axis=1)
horizon_subfunder_country = horizon_subfunder_country.sort_values('Total', ascending=False)
horizon_subfunder_country.loc['Total'] = horizon_subfunder_country.sum()
horizon_subfunder_country = horizon_subfunder_country[horizon_subfunder_country.columns[horizon_subfunder_country.loc["Total"] != 0]]

# One number per grant per country.
horizon_grant_country = horizon_recipients.pivot_table(
    index='Title', columns='recipient_country', values='recipient_funding_eur',
    aggfunc='sum', fill_value=0,
)
horizon_grant_country['Total'] = horizon_grant_country.sum(axis=1)
horizon_grant_country = horizon_grant_country.sort_values('Total', ascending=False)
horizon_grant_country.loc['Total'] = horizon_grant_country.sum()
horizon_grant_country = horizon_grant_country[horizon_grant_country.columns[horizon_grant_country.loc["Total"] != 0]]

save(horizon_subfunder_country, '14a_horizon_funding_by_subfunder_and_country')
save(horizon_grant_country, '14a_horizon_funding_by_grant_and_country')
print()
print(horizon_subfunder_country.to_string(float_format=lambda v: f'{v:,.0f}'))

print(f"\nTotal allocated to recipients: EUR "
      f"{horizon_recipients['recipient_funding_eur'].sum():,.2f}.")


167 HORIZON grants -> 944 recipient entries across 47 countries and 8 sub-funders.
74 grants have no recipient breakdown and are excluded.
Saved 14a_horizon_funding_by_subfunder_and_country -> data_output\analysis\14a_horizon_funding_by_subfunder_and_country.csv
Saved 14a_horizon_funding_by_grant_and_country -> data_output\analysis\14a_horizon_funding_by_grant_and_country.csv

recipient_country                                                                                                                          Argentina   Austria    Belgium  Bulgaria    Canada  Croatia  Cyprus   Czechia    Denmark   Estonia    Finland     France    Germany     Greece   Hungary   Iceland    Ireland    Israel      Italy    Latvia  Lebanon  Lithuania  Luxembourg   Malta  Netherlands  North Macedonia    Norway    Poland   Portugal   Romania    Serbia  Slovenia      Spain     Sweden  Switzerland  Thailand    Turkey  Ukraine  United Kingdom       Total
Sub-funder                                           

In [92]:
horizon_subfunder_country[horizon_subfunder_country.columns[horizon_subfunder_country.loc["Total"] != 0]]

recipient_country,Argentina,Austria,Belgium,Bulgaria,Canada,Croatia,Cyprus,Czechia,Denmark,Estonia,...,Serbia,Slovenia,Spain,Sweden,Switzerland,Thailand,Turkey,Ukraine,United Kingdom,Total
Sub-funder,,,,,,,,,,,,,,,,,,,,,
"Cluster 6: Food, Bioeconomy Natural Resources, Agriculture and Environment",0.00,2709888.48,12615583.00,412695.0,1025093.25,424520.0,146250.0,377280.00,12009677.66,3119949.5,...,1487341.0,927949.93,28703453.53,11826462.68,0.0,0.0,242000.0,0.0,3579171.75,1.949446e+08
European Innovation Council (EIC),0.00,4949999.00,620829.24,0.0,0.00,0.0,0.0,0.00,698002.13,0.0,...,0.0,0.00,6139407.81,6354840.00,0.0,0.0,0.0,0.0,1772209.50,4.391041e+07
"SOCIETAL CHALLENGES - Food security, sustainable agriculture and forestry, marine, maritime and inland water research, and the bioeconomy",40435.49,74738.13,776575.00,0.0,0.00,0.0,0.0,1062219.38,1607753.89,0.0,...,0.0,176998.75,4412679.03,370063.75,501685.0,97562.5,0.0,0.0,659618.75,4.385617e+07
Marie Skłodowska-Curie Actions (MSCA),0.00,528414.48,1489020.00,207295.2,0.00,0.0,0.0,0.00,1225824.48,0.0,...,0.0,0.00,1918719.68,293709.60,0.0,0.0,0.0,0.0,0.00,1.674676e+07
Widening participation and spreading excellence,0.00,0.00,609500.00,0.0,0.00,0.0,0.0,0.00,460056.25,0.0,...,0.0,0.00,89800.00,122321.25,0.0,0.0,4308057.5,897812.5,122943.75,1.044329e+07
European Research Council (ERC),0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.00,0.00,0.0,...,0.0,0.00,2355741.00,150000.00,0.0,0.0,0.0,0.0,0.00,7.882263e+06
Horizon 2020 Framework Programme,0.00,0.00,0.00,0.0,0.00,0.0,0.0,0.00,0.00,0.0,...,0.0,0.00,1185873.50,0.00,0.0,0.0,0.0,0.0,0.00,1.906961e+06
European innovation ecosystems,0.00,0.00,120325.00,0.0,0.00,0.0,0.0,0.00,0.00,131500.0,...,0.0,0.00,0.00,150531.25,0.0,0.0,0.0,0.0,0.00,1.733575e+06
Total,40435.49,8263040.09,16231832.24,619990.2,1025093.25,424520.0,146250.0,1439499.38,16001314.41,3251449.5,...,1487341.0,1104948.68,44805674.55,19267928.53,501685.0,97562.5,4550057.5,897812.5,6133943.75,3.214241e+08


#### 14b. Reconciliation: `Gov & NP contribution` vs the sum of recipient amounts

One row per grant that has a recipient breakdown, comparing the grant-level
`'Gov & NP contribution'` against the sum of its `recipient_funding_amount` entries, sorted by the
size of the gap.

`'Gov & NP contribution'` is parsed with the same space-thousands-separator rule as the recipient
amounts - at least one row is written `'11 303 944.75'`, which a plain `float()` would reject.

14a is built purely from the recipient amounts, so a gap here does **not** affect it: it flags a
problem in the grant-level column (or a genuine amendment / withdrawn partner).


In [93]:
TOLERANCE_EUR = 1.0  # at or below this, treat as rounding

_recip_sums = (horizon_recipients.groupby('grant_idx')
               .agg(n_recipients=('recipient_country', 'size'),
                    recipient_sum_eur=('recipient_funding_eur', 'sum')))

_grant_cols = ['Title', 'Sub-funder', 'Identification code', 'Year project started',
               'Gov & NP contribution', 'Total amount', 'URL for announcement']
horizon_reconciliation = horizon[_grant_cols].join(_recip_sums, how='inner')
horizon_reconciliation['gov_np_eur'] = horizon_reconciliation['Gov & NP contribution'].map(_parse_amount)
horizon_reconciliation['difference_eur'] = (horizon_reconciliation['recipient_sum_eur']
                                            - horizon_reconciliation['gov_np_eur'])
# Sort by size of gap; a missing Gov & NP sorts first rather than vanishing to the bottom as NaN.
horizon_reconciliation = (horizon_reconciliation
                          .assign(_abs=lambda d: d['difference_eur'].abs())
                          .sort_values('_abs', ascending=False, na_position='first')
                          .drop(columns='_abs'))
horizon_reconciliation.index.name = 'grant_idx'

save(horizon_reconciliation, '14b_horizon_recipient_sum_vs_gov_np')

_missing_gov = horizon_reconciliation['gov_np_eur'].isna()
_mismatch = horizon_reconciliation['difference_eur'].abs() > TOLERANCE_EUR
print(f"{len(horizon_reconciliation)} grants have a recipient breakdown "
      f"({len(horizon) - len(horizon_reconciliation)} have none and cannot be reconciled).")
print(f"{int(_mismatch.sum())} differ by more than EUR {TOLERANCE_EUR:,.0f}; "
      f"{int((~_mismatch & ~_missing_gov).sum())} agree within rounding; "
      f"{int(_missing_gov.sum())} have no 'Gov & NP contribution' to compare against.")

_show = ['Sub-funder', 'Identification code', 'Year project started', 'n_recipients',
         'gov_np_eur', 'recipient_sum_eur', 'difference_eur']
print()
print('Grants that do not reconcile:')
print(horizon_reconciliation.loc[_mismatch | _missing_gov, _show]
      .to_string(float_format=lambda v: f'{v:,.2f}'))


Saved 14b_horizon_recipient_sum_vs_gov_np -> data_output\analysis\14b_horizon_recipient_sum_vs_gov_np.csv
93 grants have a recipient breakdown (74 have none and cannot be reconciled).
0 differ by more than EUR 1; 93 agree within rounding; 0 have no 'Gov & NP contribution' to compare against.

Grants that do not reconcile:
Empty DataFrame
Columns: [Sub-funder, Identification code, Year project started, n_recipients, gov_np_eur, recipient_sum_eur, difference_eur]
Index: []


In [94]:
horizon_recipients

,grant_idx,Title,Sub-funder,recipient_institution,recipient_country,recipient_funding_eur
0,0,Like-A-Pro: From niche to mainstream - alterna...,"Cluster 6: Food, Bioeconomy Natural Resources,...",CENTRO NACIONAL DE TECNOLOGIA Y SEGURIDAD ALIM...,Spain,1341092.05
1,0,Like-A-Pro: From niche to mainstream - alterna...,"Cluster 6: Food, Bioeconomy Natural Resources,...",MOREFORSKING AS,Norway,978201.25
2,0,Like-A-Pro: From niche to mainstream - alterna...,"Cluster 6: Food, Bioeconomy Natural Resources,...",COLLABORATING CENTRE ON SUSTAINABLE CONSUMPTIO...,Germany,809937.50
3,0,Like-A-Pro: From niche to mainstream - alterna...,"Cluster 6: Food, Bioeconomy Natural Resources,...",FBCD AS,Denmark,551775.00
4,0,Like-A-Pro: From niche to mainstream - alterna...,"Cluster 6: Food, Bioeconomy Natural Resources,...",CELABOR SCRL,Belgium,531580.00
...,...,...,...,...,...,...
939,92,APRISE - Alternative Proteins Research and Inn...,Widening participation and spreading excellence,INSTITUT DE RECERCA I TECNOLOGIA AGROALIMENTARIES,Spain,89800.00
940,92,APRISE - Alternative Proteins Research and Inn...,Widening participation and spreading excellence,NUTRI BIOTECH SERVICES LTD,Malta,73243.75
941,92,APRISE - Alternative Proteins Research and Inn...,Widening participation and spreading excellence,SUTAS SUT URUNLERI AS,Turkey,59641.25
942,92,APRISE - Alternative Proteins Research and Inn...,Widening participation and spreading excellence,TURKIYE TEKNOLOJI GELISTIRME VAKFI,Turkey,40468.75
